In [ ]:
#1
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#2
!ls "/content/drive/MyDrive/New_folder"

 badodd
 badodd_class_review
 badodd_clean_11
 badodd_remapped_11
'best (1).pt'
 data.yaml
 desktop.ini
 final_11class_combined
 final_11class_split
 final_best.pt
 final_best_v8m.pt
 final_runs
 gettyimages-1501064033-640_adpp.mp4
 images
 images1
 images2
 images_clean
 labels
 labels1
 labels2
 labels_clean
 output_counted1.mp4
 output_counted2.mp4
 PICTURE1.png
 prepare_dataset.py
 README.roboflow.txt
'Recording 2.mp4'
'Recording 3.mp4'
'Recording 4.mp4'
'Recording 5.mp4'
 runs
 traffic_update.yaml
 train.ipynb
 train.py
 untitled.py
 WhatsApp10.mp4
 WhatsApp11.mp4
 WhatsApp12.mp4
 WhatsApp13.mp4
 WhatsApp14.mp4
 WhatsApp15.mp4
 WhatsApp16.mp4
 WhatsApp17.mp4
 WhatsApp1.mp4
 WhatsApp2.mp4
 WhatsApp3.mp4
 WhatsApp4.mp4
 WhatsApp5.mp4
 WhatsApp6.mp4
 WhatsApp7.mp4
 WhatsApp8.mp4
 WhatsApp9.mp4
'WhatsApp Video 2026-04-10 at 10.45.58 PM.mp4'
 yolo26n.pt
 yolov8n.pt


In [ ]:
#3
from pathlib import Path

BASE = Path("/content/drive/MyDrive/New_folder")

In [ ]:
#4
image_exts = [".jpg", ".jpeg", ".png"]

all_images = [p for p in BASE.rglob("*") if p.is_file() and p.suffix.lower() in image_exts]

print("Total images found:", len(all_images))

Total images found: 22933


In [ ]:
#5
from pathlib import Path
from collections import Counter

BASE = Path("/content/drive/MyDrive/New_folder")
image_exts = [".jpg", ".jpeg", ".png"]

all_images = [p for p in BASE.rglob("*") if p.is_file() and p.suffix.lower() in image_exts]
all_labels = [p for p in BASE.rglob("*.txt") if p.is_file()]

print("Total images:", len(all_images))
print("Total labels:", len(all_labels))

# stem = filename without extension
image_stems = {p.stem: p for p in all_images}
label_stems = {p.stem: p for p in all_labels}

images_without_labels = [p for p in all_images if p.stem not in label_stems]
labels_without_images = [p for p in all_labels if p.stem not in image_stems]

print("\nImages without labels:", len(images_without_labels))
print("Labels without images:", len(labels_without_images))

print("\nSample images without labels:")
for p in images_without_labels[:20]:
    print(p)

print("\nSample labels without images:")
for p in labels_without_images[:20]:
    print(p)

# folder-wise image count
folder_image_count = Counter(str(p.parent.relative_to(BASE)) for p in all_images)
folder_label_count = Counter(str(p.parent.relative_to(BASE)) for p in all_labels)

print("\nImage count by folder:")
for k, v in sorted(folder_image_count.items()):
    print(f"{k}: {v}")

print("\nLabel count by folder:")
for k, v in sorted(folder_label_count.items()):
    print(f"{k}: {v}")

Total images: 22933
Total labels: 22699

Images without labels: 256
Labels without images: 22

Sample images without labels:
/content/drive/MyDrive/New_folder/PICTURE1.png
/content/drive/MyDrive/New_folder/images2/Screenshot 2026-02-28 000330_png (1).rf.9MXkCC7ft0ZZknQhEJIv.png
/content/drive/MyDrive/New_folder/images2/Screenshot 2026-02-27 230518_png (1).rf.AYOZpp6PNfg1d6HmkMhg.png
/content/drive/MyDrive/New_folder/images2/Screenshot 2026-02-27 230423_png (1).rf.4ptiuA4BAJKvQxe1TzI2.png
/content/drive/MyDrive/New_folder/images2/Screenshot 2026-02-27 232818_png (1).rf.fadVTdQu9qrybvle7uM2.png
/content/drive/MyDrive/New_folder/badodd_class_review/class_0_sample_1.jpg
/content/drive/MyDrive/New_folder/badodd_class_review/class_0_sample_2.jpg
/content/drive/MyDrive/New_folder/badodd_class_review/class_0_sample_3.jpg
/content/drive/MyDrive/New_folder/badodd_class_review/class_1_sample_1.jpg
/content/drive/MyDrive/New_folder/badodd_class_review/class_1_sample_2.jpg
/content/drive/MyDrive/Ne

In [ ]:
#6
data_yaml = """
train: /content/drive/MyDrive/New_folder/final_11class_split/train/images
val: /content/drive/MyDrive/New_folder/final_11class_split/valid/images

nc: 11
names: ['Ambulance', 'Auto_ricksaw', 'Bike', 'Bus', 'Fire_truck', 'Leguna', 'Pickup', 'Police_car', 'Private_car', 'Truck', 'cng']
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)

print("YAML ready")

YAML ready


In [ ]:
#7
### USING YOLOV8M MODEL TO RETRAIN
!pip install -q ultralytics

# --- Code to generate data.yaml added here ---
data_yaml = """
train: /content/drive/MyDrive/New_folder/final_11class_split/train/images
val: /content/drive/MyDrive/New_folder/final_11class_split/valid/images

nc: 11
names: ['Ambulance', 'Auto_ricksaw', 'Bike', 'Bus', 'Fire_truck', 'Leguna', 'Pickup', 'Police_car', 'Private_car', 'Truck', 'cng']
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)
# --- End of added code ---

from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="/content/data.yaml",
    epochs=100,
    imgsz=960,
    batch=8,
    device=0,

    # stability
    patience=30,

    # optimizer
    optimizer="AdamW",
    lr0=0.0005,

    # augmentation (VERY IMPORTANT)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    mixup=0.2,
    fliplr=0.5,

    project="/content/drive/MyDrive/New_folder/final_runs",
    name="final_v8m_model"
)

In [ ]:
#8
!cp "/content/drive/MyDrive/New_folder/final_runs/final_v8m_model-2/weights/best.pt" \
    "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

In [ ]:
#9
!pip install -q ultralytics

from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/New_folder/final_best_v8m.pt")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
## final_best_v8m - 2,15 UPDATED SAFE EMV VERSION

!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque

# -----------------------------------
# 1. SETTINGS
# -----------------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO1_PATH = "/content/WhatsApp15.mp4"
VIDEO2_PATH = "/content/WhatsApp2.mp4"
OUTPUT_PATH = "/content/two_lane_emv_locked_signal_SAFE.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

# For defense, keep only Ambulance if your demo focuses on ambulance priority
emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

NORMAL_HOLD_FRAMES = 30
EMV_LOCK_FRAMES = 45
PCU_SMOOTH_WINDOW = 10

# safer EMV filter
EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4

# -----------------------------------
# 2. OPEN VIDEOS
# -----------------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 90

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------------
# 3. STATE MEMORY
# -----------------------------------
current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
hold_counter = 0

emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

# EMV confirmation memory
lane1_emv_confirm = 0
lane2_emv_confirm = 0

# -----------------------------------
# 4. LANE PROCESSOR
# -----------------------------------
def process_lane(frame):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(frame, conf=CONF_THRES, imgsz=960, verbose=False)

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        h, w = frame.shape[:2]

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)
            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]
            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            # safer EMV candidate rule
            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------------
# 5. MAIN LOOP
# -----------------------------------
frame_idx = 0

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2)

    # EMV multi-frame confirmation
    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else 0
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else 0

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    # -----------------------------------
    # 6. DECISION ENGINE
    # -----------------------------------
    decision = ""

    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True

        if emv1 and not emv2:
            emv_locked_lane = "LANE 1"
        elif emv2 and not emv1:
            emv_locked_lane = "LANE 2"
        else:
            d1 = emv_dist1 if emv_dist1 is not None else 1e9
            d2 = emv_dist2 if emv_dist2 is not None else 1e9
            emv_locked_lane = "LANE 1" if d1 <= d2 else "LANE 2"

        current_green_lane = emv_locked_lane
        emv_lock_counter = EMV_LOCK_FRAMES
        decision = f"CONFIRMED EMV -> CLEAR {current_green_lane}"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        current_green_lane = emv_locked_lane
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane}"
        emv_lock_counter -= 1

    else:
        emv_locked = False
        emv_locked_lane = None

        if hold_counter <= 0:
            current_mode = "NORMAL PCU MODE"

            if smooth_pcu1 > smooth_pcu2:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1
            elif smooth_pcu2 > smooth_pcu1:
                current_green_lane = "LANE 2"
                max_pcu = smooth_pcu2
            else:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1

            if max_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif max_pcu < 5:
                decision = f"GREEN -> {current_green_lane} for 10s"
            elif max_pcu < 10:
                decision = f"GREEN -> {current_green_lane} for 20s"
            else:
                decision = f"GREEN -> {current_green_lane} for 30s"

            hold_counter = NORMAL_HOLD_FRAMES
        else:
            current_mode = "NORMAL PCU MODE"
            decision = f"HOLD GREEN -> {current_green_lane}"
            hold_counter -= 1

    # -----------------------------------
    # 7. DRAW SIGNAL STATE
    # -----------------------------------
    def draw_signal_panel(frame, lane_name, is_green, pcu_value, count_value, emv_flag, emv_confirm_count):
        signal_x = 20
        signal_y = 20

        red_color = (0, 0, 255) if not is_green else (60, 60, 60)
        green_color = (0, 255, 0) if is_green else (60, 60, 60)

        cv2.circle(frame, (signal_x, signal_y), 12, red_color, -1)
        cv2.circle(frame, (signal_x + 35, signal_y), 12, green_color, -1)

        cv2.putText(frame, lane_name, (60, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Count: {count_value}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"EMV: {'YES' if emv_flag else 'NO'} ({emv_confirm_count})", (20, 105),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    lane1_count = sum(counts1.values())
    lane2_count = sum(counts2.values())

    draw_signal_panel(frame1, "LANE 1", current_green_lane == "LANE 1",
                      smooth_pcu1, lane1_count, emv1, lane1_emv_confirm)

    draw_signal_panel(frame2, "LANE 2", current_green_lane == "LANE 2",
                      smooth_pcu2, lane2_count, emv2, lane2_emv_confirm)

    if current_green_lane == "LANE 1":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 255), 5)
    else:
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 255), 5)

    # -----------------------------------
    # 8. COMBINE OUTPUT
    # -----------------------------------
    top = np.zeros((90, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)
    cv2.putText(top, f"MODE: {current_mode}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)
    cv2.putText(top, decision, (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

print("Done. Saved:", OUTPUT_PATH)

In [ ]:
## final_best_v8m - 16,15 UPDATED SAFE EMV VERSION

!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp16.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque

# -----------------------------------
# 1. SETTINGS
# -----------------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO1_PATH = "/content/WhatsApp15.mp4"
VIDEO2_PATH = "/content/WhatsApp16.mp4"
OUTPUT_PATH = "/content/two_lane_emv_locked_signal_SAFE.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

# For defense, keep only Ambulance if your demo focuses on ambulance priority
emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

NORMAL_HOLD_FRAMES = 30
EMV_LOCK_FRAMES = 45
PCU_SMOOTH_WINDOW = 10

# safer EMV filter
EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4

# -----------------------------------
# 2. OPEN VIDEOS
# -----------------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 90

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------------
# 3. STATE MEMORY
# -----------------------------------
current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
hold_counter = 0

emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

# EMV confirmation memory
lane1_emv_confirm = 0
lane2_emv_confirm = 0

# -----------------------------------
# 4. LANE PROCESSOR
# -----------------------------------
def process_lane(frame):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(frame, conf=CONF_THRES, imgsz=960, verbose=False)

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        h, w = frame.shape[:2]

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)
            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]
            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            # safer EMV candidate rule
            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------------
# 5. MAIN LOOP
# -----------------------------------
frame_idx = 0

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2)

    # EMV multi-frame confirmation
    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else 0
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else 0

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    # -----------------------------------
    # 6. DECISION ENGINE
    # -----------------------------------
    decision = ""

    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True

        if emv1 and not emv2:
            emv_locked_lane = "LANE 1"
        elif emv2 and not emv1:
            emv_locked_lane = "LANE 2"
        else:
            d1 = emv_dist1 if emv_dist1 is not None else 1e9
            d2 = emv_dist2 if emv_dist2 is not None else 1e9
            emv_locked_lane = "LANE 1" if d1 <= d2 else "LANE 2"

        current_green_lane = emv_locked_lane
        emv_lock_counter = EMV_LOCK_FRAMES
        decision = f"CONFIRMED EMV -> CLEAR {current_green_lane}"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        current_green_lane = emv_locked_lane
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane}"
        emv_lock_counter -= 1

    else:
        emv_locked = False
        emv_locked_lane = None

        if hold_counter <= 0:
            current_mode = "NORMAL PCU MODE"

            if smooth_pcu1 > smooth_pcu2:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1
            elif smooth_pcu2 > smooth_pcu1:
                current_green_lane = "LANE 2"
                max_pcu = smooth_pcu2
            else:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1

            if max_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif max_pcu < 5:
                decision = f"GREEN -> {current_green_lane} for 10s"
            elif max_pcu < 10:
                decision = f"GREEN -> {current_green_lane} for 20s"
            else:
                decision = f"GREEN -> {current_green_lane} for 30s"

            hold_counter = NORMAL_HOLD_FRAMES
        else:
            current_mode = "NORMAL PCU MODE"
            decision = f"HOLD GREEN -> {current_green_lane}"
            hold_counter -= 1

    # -----------------------------------
    # 7. DRAW SIGNAL STATE
    # -----------------------------------
    def draw_signal_panel(frame, lane_name, is_green, pcu_value, count_value, emv_flag, emv_confirm_count):
        signal_x = 20
        signal_y = 20

        red_color = (0, 0, 255) if not is_green else (60, 60, 60)
        green_color = (0, 255, 0) if is_green else (60, 60, 60)

        cv2.circle(frame, (signal_x, signal_y), 12, red_color, -1)
        cv2.circle(frame, (signal_x + 35, signal_y), 12, green_color, -1)

        cv2.putText(frame, lane_name, (60, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Count: {count_value}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"EMV: {'YES' if emv_flag else 'NO'} ({emv_confirm_count})", (20, 105),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    lane1_count = sum(counts1.values())
    lane2_count = sum(counts2.values())

    draw_signal_panel(frame1, "LANE 1", current_green_lane == "LANE 1",
                      smooth_pcu1, lane1_count, emv1, lane1_emv_confirm)

    draw_signal_panel(frame2, "LANE 2", current_green_lane == "LANE 2",
                      smooth_pcu2, lane2_count, emv2, lane2_emv_confirm)

    if current_green_lane == "LANE 1":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 255), 5)
    else:
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 255), 5)

    # -----------------------------------
    # 8. COMBINE OUTPUT
    # -----------------------------------
    top = np.zeros((90, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)
    cv2.putText(top, f"MODE: {current_mode}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)
    cv2.putText(top, decision, (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

print("Done. Saved:", OUTPUT_PATH)

In [ ]:
## final_best_v8m - 8,15 UPDATED SAFE EMV VERSION

!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp8.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque

# -----------------------------------
# 1. SETTINGS
# -----------------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO1_PATH = "/content/WhatsApp15.mp4"
VIDEO2_PATH = "/content/WhatsApp8.mp4"
OUTPUT_PATH = "/content/two_lane_emv_locked_signal_SAFE.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

# For defense, keep only Ambulance if your demo focuses on ambulance priority
emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

NORMAL_HOLD_FRAMES = 30
EMV_LOCK_FRAMES = 45
PCU_SMOOTH_WINDOW = 10

# safer EMV filter
EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4

# -----------------------------------
# 2. OPEN VIDEOS
# -----------------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 90

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------------
# 3. STATE MEMORY
# -----------------------------------
current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
hold_counter = 0

emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

# EMV confirmation memory
lane1_emv_confirm = 0
lane2_emv_confirm = 0

# -----------------------------------
# 4. LANE PROCESSOR
# -----------------------------------
def process_lane(frame):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(frame, conf=CONF_THRES, imgsz=960, verbose=False)

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        h, w = frame.shape[:2]

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)
            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]
            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            # safer EMV candidate rule
            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------------
# 5. MAIN LOOP
# -----------------------------------
frame_idx = 0

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2)

    # EMV multi-frame confirmation
    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else 0
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else 0

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    # -----------------------------------
       # -----------------------------------
    # 6. DECISION ENGINE
    # -----------------------------------
    decision = ""

    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True

        # trigger alarm ONLY ONCE
        if not alarm_triggered:
            os.system("printf '\\a'")   # beep sound
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        if emv1 and not emv2:
            emv_locked_lane = "LANE 1"
        elif emv2 and not emv1:
            emv_locked_lane = "LANE 2"
        else:
            d1 = emv_dist1 if emv_dist1 is not None else 1e9
            d2 = emv_dist2 if emv_dist2 is not None else 1e9
            emv_locked_lane = "LANE 1" if d1 <= d2 else "LANE 2"

        current_green_lane = emv_locked_lane
        emv_lock_counter = EMV_LOCK_FRAMES
        decision = f"EMV DETECTED -> CLEAR {current_green_lane}"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        current_green_lane = emv_locked_lane
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane}"
        emv_lock_counter -= 1

    else:
        emv_locked = False
        emv_locked_lane = None
        alarm_triggered = False

        if hold_counter <= 0:
            current_mode = "NORMAL PCU MODE"

            if smooth_pcu1 > smooth_pcu2:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1
            elif smooth_pcu2 > smooth_pcu1:
                current_green_lane = "LANE 2"
                max_pcu = smooth_pcu2
            else:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1

            if max_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif max_pcu < 5:
                decision = f"GREEN -> {current_green_lane} for 10s"
            elif max_pcu < 10:
                decision = f"GREEN -> {current_green_lane} for 20s"
            else:
                decision = f"GREEN -> {current_green_lane} for 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            current_mode = "NORMAL PCU MODE"
            decision = f"HOLD GREEN -> {current_green_lane}"
            hold_counter -= 1

    # -----------------------------------
    # 7. DRAW SIGNAL STATE
    # -----------------------------------
    def draw_signal_panel(frame, lane_name, is_green, pcu_value, count_value, emv_flag, emv_confirm_count):
        signal_x = 20
        signal_y = 20

        red_color = (0, 0, 255) if not is_green else (60, 60, 60)
        green_color = (0, 255, 0) if is_green else (60, 60, 60)

        cv2.circle(frame, (signal_x, signal_y), 12, red_color, -1)
        cv2.circle(frame, (signal_x + 35, signal_y), 12, green_color, -1)

        cv2.putText(frame, lane_name, (60, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Count: {count_value}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"EMV: {'YES' if emv_flag else 'NO'} ({emv_confirm_count})", (20, 105),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    lane1_count = sum(counts1.values())
    lane2_count = sum(counts2.values())

    draw_signal_panel(frame1, "LANE 1", current_green_lane == "LANE 1",
                      smooth_pcu1, lane1_count, emv1, lane1_emv_confirm)

    draw_signal_panel(frame2, "LANE 2", current_green_lane == "LANE 2",
                      smooth_pcu2, lane2_count, emv2, lane2_emv_confirm)

    if current_green_lane == "LANE 1":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 255), 5)
    else:
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 255), 5)

    # -----------------------------------
    # 8. COMBINE OUTPUT
    # -----------------------------------
    top = np.zeros((90, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)
    cv2.putText(top, f"MODE: {current_mode}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)
    cv2.putText(top, decision, (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

print("Done. Saved:", OUTPUT_PATH)

In [ ]:
## final_best_v8m - 8,15 UPDATED SAFE EMV VERSION

!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp16.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque

# -----------------------------------
# 1. SETTINGS
# -----------------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO1_PATH = "/content/WhatsApp15.mp4"
VIDEO2_PATH = "/content/WhatsApp16.mp4"
OUTPUT_PATH = "/content/two_lane_emv_locked_signal_SAFE.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

# For defense, keep only Ambulance if your demo focuses on ambulance priority
emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

NORMAL_HOLD_FRAMES = 30
EMV_LOCK_FRAMES = 45
PCU_SMOOTH_WINDOW = 10

# safer EMV filter
EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4

# -----------------------------------
# 2. OPEN VIDEOS
# -----------------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 90

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------------
# 3. STATE MEMORY
# -----------------------------------
current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
hold_counter = 0

emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

# EMV confirmation memory
lane1_emv_confirm = 0
lane2_emv_confirm = 0

# -----------------------------------
# 4. LANE PROCESSOR
# -----------------------------------
def process_lane(frame):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(frame, conf=CONF_THRES, imgsz=960, verbose=False)

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        h, w = frame.shape[:2]

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)
            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]
            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            # safer EMV candidate rule
            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------------
# 5. MAIN LOOP
# -----------------------------------
frame_idx = 0

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2)

    # EMV multi-frame confirmation
    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else 0
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else 0

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    # -----------------------------------
       # -----------------------------------
    # 6. DECISION ENGINE
    # -----------------------------------
    decision = ""

    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True

        # trigger alarm ONLY ONCE
        if not alarm_triggered:
            os.system("printf '\\a'")   # beep sound
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        if emv1 and not emv2:
            emv_locked_lane = "LANE 1"
        elif emv2 and not emv1:
            emv_locked_lane = "LANE 2"
        else:
            d1 = emv_dist1 if emv_dist1 is not None else 1e9
            d2 = emv_dist2 if emv_dist2 is not None else 1e9
            emv_locked_lane = "LANE 1" if d1 <= d2 else "LANE 2"

        current_green_lane = emv_locked_lane
        emv_lock_counter = EMV_LOCK_FRAMES
        decision = f"EMV DETECTED -> CLEAR {current_green_lane}"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        current_green_lane = emv_locked_lane
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane}"
        emv_lock_counter -= 1

    else:
        emv_locked = False
        emv_locked_lane = None
        alarm_triggered = False

        if hold_counter <= 0:
            current_mode = "NORMAL PCU MODE"

            if smooth_pcu1 > smooth_pcu2:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1
            elif smooth_pcu2 > smooth_pcu1:
                current_green_lane = "LANE 2"
                max_pcu = smooth_pcu2
            else:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1

            if max_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif max_pcu < 5:
                decision = f"GREEN -> {current_green_lane} for 10s"
            elif max_pcu < 10:
                decision = f"GREEN -> {current_green_lane} for 20s"
            else:
                decision = f"GREEN -> {current_green_lane} for 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            current_mode = "NORMAL PCU MODE"
            decision = f"HOLD GREEN -> {current_green_lane}"
            hold_counter -= 1

    # -----------------------------------
    # 7. DRAW SIGNAL STATE
    # -----------------------------------
    def draw_signal_panel(frame, lane_name, is_green, pcu_value, count_value, emv_flag, emv_confirm_count):
        signal_x = 20
        signal_y = 20

        red_color = (0, 0, 255) if not is_green else (60, 60, 60)
        green_color = (0, 255, 0) if is_green else (60, 60, 60)

        cv2.circle(frame, (signal_x, signal_y), 12, red_color, -1)
        cv2.circle(frame, (signal_x + 35, signal_y), 12, green_color, -1)

        cv2.putText(frame, lane_name, (60, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Count: {count_value}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"EMV: {'YES' if emv_flag else 'NO'} ({emv_confirm_count})", (20, 105),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    lane1_count = sum(counts1.values())
    lane2_count = sum(counts2.values())

    draw_signal_panel(frame1, "LANE 1", current_green_lane == "LANE 1",
                      smooth_pcu1, lane1_count, emv1, lane1_emv_confirm)

    draw_signal_panel(frame2, "LANE 2", current_green_lane == "LANE 2",
                      smooth_pcu2, lane2_count, emv2, lane2_emv_confirm)

    if current_green_lane == "LANE 1":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 255), 5)
    else:
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 255), 5)

    # -----------------------------------
    # 8. COMBINE OUTPUT
    # -----------------------------------
    top = np.zeros((90, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)
    cv2.putText(top, f"MODE: {current_mode}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)
    cv2.putText(top, decision, (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

print("Done. Saved:", OUTPUT_PATH)

In [ ]:
from google.colab import files
files.download("/content/two_lane_emv_locked_signal_SAFE.mp4")

In [ ]:
## final_best_v8m - 8,15 UPDATED SAFE EMV VERSION

!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp8.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque

# -----------------------------------
# 1. SETTINGS
# -----------------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO1_PATH = "/content/WhatsApp2.mp4"
VIDEO2_PATH = "/content/WhatsApp8.mp4"
OUTPUT_PATH = "/content/two_lane_emv_locked_signal_SAFE.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

# For defense, keep only Ambulance if your demo focuses on ambulance priority
emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

NORMAL_HOLD_FRAMES = 30
EMV_LOCK_FRAMES = 45
PCU_SMOOTH_WINDOW = 10

# safer EMV filter
EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4

# -----------------------------------
# 2. OPEN VIDEOS
# -----------------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 90

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------------
# 3. STATE MEMORY
# -----------------------------------
current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
hold_counter = 0

emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

# EMV confirmation memory
lane1_emv_confirm = 0
lane2_emv_confirm = 0

# -----------------------------------
# 4. LANE PROCESSOR
# -----------------------------------
def process_lane(frame):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(frame, conf=CONF_THRES, imgsz=960, verbose=False)

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        h, w = frame.shape[:2]

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)
            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]
            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            # safer EMV candidate rule
            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------------
# 5. MAIN LOOP
# -----------------------------------
frame_idx = 0

while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2)

    # EMV multi-frame confirmation
    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else 0
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else 0

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    # -----------------------------------
       # -----------------------------------
    # 6. DECISION ENGINE
    # -----------------------------------
    decision = ""

    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True

        # trigger alarm ONLY ONCE
        if not alarm_triggered:
            os.system("printf '\\a'")   # beep sound
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        if emv1 and not emv2:
            emv_locked_lane = "LANE 1"
        elif emv2 and not emv1:
            emv_locked_lane = "LANE 2"
        else:
            d1 = emv_dist1 if emv_dist1 is not None else 1e9
            d2 = emv_dist2 if emv_dist2 is not None else 1e9
            emv_locked_lane = "LANE 1" if d1 <= d2 else "LANE 2"

        current_green_lane = emv_locked_lane
        emv_lock_counter = EMV_LOCK_FRAMES
        decision = f"EMV DETECTED -> CLEAR {current_green_lane}"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        current_green_lane = emv_locked_lane
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane}"
        emv_lock_counter -= 1

    else:
        emv_locked = False
        emv_locked_lane = None
        alarm_triggered = False

        if hold_counter <= 0:
            current_mode = "NORMAL PCU MODE"

            if smooth_pcu1 > smooth_pcu2:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1
            elif smooth_pcu2 > smooth_pcu1:
                current_green_lane = "LANE 2"
                max_pcu = smooth_pcu2
            else:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1

            if max_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif max_pcu < 5:
                decision = f"GREEN -> {current_green_lane} for 10s"
            elif max_pcu < 10:
                decision = f"GREEN -> {current_green_lane} for 20s"
            else:
                decision = f"GREEN -> {current_green_lane} for 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            current_mode = "NORMAL PCU MODE"
            decision = f"HOLD GREEN -> {current_green_lane}"
            hold_counter -= 1

    # -----------------------------------
    # 7. DRAW SIGNAL STATE
    # -----------------------------------
    def draw_signal_panel(frame, lane_name, is_green, pcu_value, count_value, emv_flag, emv_confirm_count):
        signal_x = 20
        signal_y = 20

        red_color = (0, 0, 255) if not is_green else (60, 60, 60)
        green_color = (0, 255, 0) if is_green else (60, 60, 60)

        cv2.circle(frame, (signal_x, signal_y), 12, red_color, -1)
        cv2.circle(frame, (signal_x + 35, signal_y), 12, green_color, -1)

        cv2.putText(frame, lane_name, (60, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Count: {count_value}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"EMV: {'YES' if emv_flag else 'NO'} ({emv_confirm_count})", (20, 105),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    lane1_count = sum(counts1.values())
    lane2_count = sum(counts2.values())

    draw_signal_panel(frame1, "LANE 1", current_green_lane == "LANE 1",
                      smooth_pcu1, lane1_count, emv1, lane1_emv_confirm)

    draw_signal_panel(frame2, "LANE 2", current_green_lane == "LANE 2",
                      smooth_pcu2, lane2_count, emv2, lane2_emv_confirm)

    if current_green_lane == "LANE 1":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 255), 5)
    else:
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 255), 5)

    # -----------------------------------
    # 8. COMBINE OUTPUT
    # -----------------------------------
    top = np.zeros((90, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)
    cv2.putText(top, f"MODE: {current_mode}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)
    cv2.putText(top, decision, (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

print("Done. Saved:", OUTPUT_PATH)

In [ ]:
# FINAL ONE-VIDEO CODE: Vehicle Detection + Distance + PCU + Ambulance Alert

!pip install -q ultralytics
!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/

from ultralytics import YOLO
import cv2
import os
from collections import defaultdict, deque

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO_PATH = "/content/WhatsApp2.mp4"
OUTPUT_PATH = "/content/final_one_video_emv_pcu.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4
EMV_LOCK_FRAMES = 45

PCU_SMOOTH_WINDOW = 10
NORMAL_HOLD_FRAMES = 30

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. VIDEO
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 10

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

emv_confirm = 0
emv_locked = False
emv_lock_counter = 0
alarm_triggered = False

hold_counter = 0
current_mode = "NORMAL PCU MODE"

frame_idx = 0

# -----------------------------
# 4. MAIN LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)  # normal vehicle = green
            if is_emv_candidate:
                color = (0, 165, 255)  # ambulance candidate = orange

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    # -----------------------------
    # 5. PCU + EMV CONFIRMATION
    # -----------------------------
    pcu_history.append(lane_pcu)
    smooth_pcu = sum(pcu_history) / max(1, len(pcu_history))

    if emv_candidate:
        emv_confirm += 1
    else:
        emv_confirm = 0

    emv_detected = emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 6. DECISION ENGINE
    # -----------------------------
    if emv_detected:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        if not alarm_triggered:
            os.system("printf '\\a'")
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        decision = "EMV DETECTED -> CLEAR ROAD"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        decision = "LOCKED EMV PRIORITY"
        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        alarm_triggered = False

        if hold_counter <= 0:
            if smooth_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif smooth_pcu < 5:
                decision = "LOW TRAFFIC -> GREEN 10s"
            elif smooth_pcu < 10:
                decision = "MEDIUM TRAFFIC -> GREEN 20s"
            else:
                decision = "HIGH TRAFFIC -> GREEN 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            decision = "HOLD GREEN"
            hold_counter -= 1

    # -----------------------------
    # 7. DISPLAY
    # -----------------------------
    total_count = sum(lane_counts.values())

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    cv2.putText(frame, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, decision, (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, f"PCU: {smooth_pcu:.1f}", (20, 105),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {total_count}", (20, 135),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.putText(frame, f"EMV Confirm: {emv_confirm}/{EMV_CONFIRM_FRAMES}", (20, 165),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    if emv_detected and nearest_emv_distance is not None:
        cv2.putText(frame, f"EMV Distance: {nearest_emv_distance:.1f}m", (20, 195),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    y_text = 230
    for cls_name, cnt in sorted(lane_counts.items()):
        cv2.putText(frame, f"{cls_name}: {cnt}", (20, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        y_text += 25

    out.write(frame)

cap.release()
out.release()

print("Done. Saved:", OUTPUT_PATH)
print("Meters per pixel used:", meters_per_pixel)

In [ ]:
from google.colab import files
files.download("/content/final_one_video_emv_pcu.mp4")

In [ ]:
# FINAL ONE-VIDEO CODE: Vehicle Detection + Distance + PCU + Ambulance Alert

!pip install -q ultralytics
!cp "/content/drive/MyDrive/New_folder/WhatsApp3.mp4" /content/

from ultralytics import YOLO
import cv2
import os
from collections import defaultdict, deque

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO_PATH = "/content/WhatsApp3.mp4"
OUTPUT_PATH = "/content/final_one_video_emv_pcu.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4
EMV_LOCK_FRAMES = 45

PCU_SMOOTH_WINDOW = 10
NORMAL_HOLD_FRAMES = 30

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. VIDEO
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 10

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

emv_confirm = 0
emv_locked = False
emv_lock_counter = 0
alarm_triggered = False

hold_counter = 0
current_mode = "NORMAL PCU MODE"

frame_idx = 0

# -----------------------------
# 4. MAIN LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)  # normal vehicle = green
            if is_emv_candidate:
                color = (0, 165, 255)  # ambulance candidate = orange

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    # -----------------------------
    # 5. PCU + EMV CONFIRMATION
    # -----------------------------
    pcu_history.append(lane_pcu)
    smooth_pcu = sum(pcu_history) / max(1, len(pcu_history))

    if emv_candidate:
        emv_confirm += 1
    else:
        emv_confirm = 0

    emv_detected = emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 6. DECISION ENGINE
    # -----------------------------
    if emv_detected:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        if not alarm_triggered:
            os.system("printf '\\a'")
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        decision = "EMV DETECTED -> CLEAR ROAD"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        decision = "LOCKED EMV PRIORITY"
        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        alarm_triggered = False

        if hold_counter <= 0:
            if smooth_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif smooth_pcu < 5:
                decision = "LOW TRAFFIC -> GREEN 10s"
            elif smooth_pcu < 10:
                decision = "MEDIUM TRAFFIC -> GREEN 20s"
            else:
                decision = "HIGH TRAFFIC -> GREEN 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            decision = "HOLD GREEN"
            hold_counter -= 1

    # -----------------------------
    # 7. DISPLAY
    # -----------------------------
    total_count = sum(lane_counts.values())

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    cv2.putText(frame, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, decision, (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, f"PCU: {smooth_pcu:.1f}", (20, 105),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {total_count}", (20, 135),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.putText(frame, f"EMV Confirm: {emv_confirm}/{EMV_CONFIRM_FRAMES}", (20, 165),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    if emv_detected and nearest_emv_distance is not None:
        cv2.putText(frame, f"EMV Distance: {nearest_emv_distance:.1f}m", (20, 195),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    y_text = 230
    for cls_name, cnt in sorted(lane_counts.items()):
        cv2.putText(frame, f"{cls_name}: {cnt}", (20, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        y_text += 25

    out.write(frame)

cap.release()
out.release()

print("Done. Saved:", OUTPUT_PATH)
print("Meters per pixel used:", meters_per_pixel)

In [ ]:
###

# FINAL ONE-VIDEO CODE:
# Vehicle Detection + Distance + PCU + Ambulance Alert + Alarm Sound in Downloaded Video

!pip install -q ultralytics moviepy scipy
!cp "/content/drive/MyDrive/New_folder/WhatsApp3.mp4" /content/

from ultralytics import YOLO
import cv2
import os
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO_PATH = "/content/WhatsApp3.mp4"
OUTPUT_PATH = "/content/final_one_video_emv_pcu_silent.mp4"
FINAL_OUTPUT_PATH = "/content/final_one_video_emv_pcu_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4
EMV_LOCK_FRAMES = 45

PCU_SMOOTH_WINDOW = 10
NORMAL_HOLD_FRAMES = 30

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. VIDEO
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 10

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

emv_confirm = 0
emv_locked = False
emv_lock_counter = 0
alarm_triggered = False

hold_counter = 0
current_mode = "NORMAL PCU MODE"
frame_idx = 0

# alarm time memory
alarm_segments = []
alarm_active = False
alarm_start_time = None

# -----------------------------
# 4. MAIN LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    # -----------------------------
    # 5. PCU + EMV CONFIRMATION
    # -----------------------------
    pcu_history.append(lane_pcu)
    smooth_pcu = sum(pcu_history) / max(1, len(pcu_history))

    if emv_candidate:
        emv_confirm += 1
    else:
        emv_confirm = 0

    emv_detected = emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 6. DECISION ENGINE + ALARM SEGMENTS
    # -----------------------------
    if emv_detected:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        current_time = frame_idx / fps

        if not alarm_active:
            alarm_start_time = current_time
            alarm_active = True

        if not alarm_triggered:
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        decision = "EMV DETECTED -> CLEAR ROAD"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        decision = "LOCKED EMV PRIORITY"
        emv_lock_counter -= 1

        current_time = frame_idx / fps
        if not alarm_active:
            alarm_start_time = current_time
            alarm_active = True

    else:
        if alarm_active:
            alarm_end_time = frame_idx / fps
            alarm_segments.append((alarm_start_time, alarm_end_time))
            alarm_active = False
            alarm_start_time = None

        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        alarm_triggered = False

        if hold_counter <= 0:
            if smooth_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif smooth_pcu < 5:
                decision = "LOW TRAFFIC -> GREEN 10s"
            elif smooth_pcu < 10:
                decision = "MEDIUM TRAFFIC -> GREEN 20s"
            else:
                decision = "HIGH TRAFFIC -> GREEN 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            decision = "HOLD GREEN"
            hold_counter -= 1

    # -----------------------------
    # 7. DISPLAY
    # -----------------------------
    total_count = sum(lane_counts.values())
    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        cv2.rectangle(frame, (0, 0), (w, 90), (0, 0, 255), -1)
        cv2.putText(frame, "EMERGENCY ALERT - CLEAR ROAD",
                    (20, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (255, 255, 255),
                    3)

    cv2.putText(frame, f"MODE: {current_mode}", (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, decision, (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, f"PCU: {smooth_pcu:.1f}", (20, 190),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {total_count}", (20, 220),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.putText(frame, f"EMV Confirm: {emv_confirm}/{EMV_CONFIRM_FRAMES}", (20, 250),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    if emv_detected and nearest_emv_distance is not None:
        cv2.putText(frame, f"EMV Distance: {nearest_emv_distance:.1f}m", (20, 280),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    y_text = 315
    for cls_name, cnt in sorted(lane_counts.items()):
        cv2.putText(frame, f"{cls_name}: {cnt}", (20, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        y_text += 25

    out.write(frame)

cap.release()
out.release()

# If video ended while alarm still active
if alarm_active:
    alarm_segments.append((alarm_start_time, frame_idx / fps))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)
write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD ALARM AUDIO ONLY DURING EMV SEGMENTS
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 1.0

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

# -----------------------------
# 10. DOWNLOAD
# -----------------------------
from google.colab import files
files.download(FINAL_OUTPUT_PATH)

In [ ]:
###

# FINAL ONE-VIDEO CODE:
# Vehicle Detection + Distance + PCU + Ambulance Alert + Alarm Sound in Downloaded Video

!pip install -q ultralytics moviepy scipy
!cp "/content/drive/MyDrive/New_folder/WhatsApp4.mp4" /content/

from ultralytics import YOLO
import cv2
import os
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO_PATH = "/content/WhatsApp4.mp4"
OUTPUT_PATH = "/content/final_one_video_emv_pcu_silent.mp4"
FINAL_OUTPUT_PATH = "/content/final_one_video_emv_pcu_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4
EMV_LOCK_FRAMES = 45

PCU_SMOOTH_WINDOW = 10
NORMAL_HOLD_FRAMES = 30

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. VIDEO
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 10

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

emv_confirm = 0
emv_locked = False
emv_lock_counter = 0
alarm_triggered = False

hold_counter = 0
current_mode = "NORMAL PCU MODE"
frame_idx = 0

# alarm time memory
alarm_segments = []
alarm_active = False
alarm_start_time = None

# -----------------------------
# 4. MAIN LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    # -----------------------------
    # 5. PCU + EMV CONFIRMATION
    # -----------------------------
    pcu_history.append(lane_pcu)
    smooth_pcu = sum(pcu_history) / max(1, len(pcu_history))

    if emv_candidate:
        emv_confirm += 1
    else:
        emv_confirm = 0

    emv_detected = emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 6. DECISION ENGINE + ALARM SEGMENTS
    # -----------------------------
    if emv_detected:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        current_time = frame_idx / fps

        if not alarm_active:
            alarm_start_time = current_time
            alarm_active = True

        if not alarm_triggered:
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        decision = "EMV DETECTED -> CLEAR ROAD"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        decision = "LOCKED EMV PRIORITY"
        emv_lock_counter -= 1

        current_time = frame_idx / fps
        if not alarm_active:
            alarm_start_time = current_time
            alarm_active = True

    else:
        if alarm_active:
            alarm_end_time = frame_idx / fps
            alarm_segments.append((alarm_start_time, alarm_end_time))
            alarm_active = False
            alarm_start_time = None

        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        alarm_triggered = False

        if hold_counter <= 0:
            if smooth_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif smooth_pcu < 5:
                decision = "LOW TRAFFIC -> GREEN 10s"
            elif smooth_pcu < 10:
                decision = "MEDIUM TRAFFIC -> GREEN 20s"
            else:
                decision = "HIGH TRAFFIC -> GREEN 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            decision = "HOLD GREEN"
            hold_counter -= 1

    # -----------------------------
    # 7. DISPLAY
    # -----------------------------
    total_count = sum(lane_counts.values())
    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        cv2.rectangle(frame, (0, 0), (w, 90), (0, 0, 255), -1)
        cv2.putText(frame, "EMERGENCY ALERT - CLEAR ROAD",
                    (20, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (255, 255, 255),
                    3)

    cv2.putText(frame, f"MODE: {current_mode}", (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, decision, (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, f"PCU: {smooth_pcu:.1f}", (20, 190),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {total_count}", (20, 220),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.putText(frame, f"EMV Confirm: {emv_confirm}/{EMV_CONFIRM_FRAMES}", (20, 250),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    if emv_detected and nearest_emv_distance is not None:
        cv2.putText(frame, f"EMV Distance: {nearest_emv_distance:.1f}m", (20, 280),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    y_text = 315
    for cls_name, cnt in sorted(lane_counts.items()):
        cv2.putText(frame, f"{cls_name}: {cnt}", (20, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        y_text += 25

    out.write(frame)

cap.release()
out.release()

# If video ended while alarm still active
if alarm_active:
    alarm_segments.append((alarm_start_time, frame_idx / fps))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)
write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD ALARM AUDIO ONLY DURING EMV SEGMENTS
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 1.0

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

# -----------------------------
# 10. DOWNLOAD
# -----------------------------
from google.colab import files
files.download(FINAL_OUTPUT_PATH)

EMERGENCY VEHICLE DETECTED - CLEAR ROAD
EMERGENCY VEHICLE DETECTED - CLEAR ROAD
Silent video saved: /content/final_one_video_emv_pcu_silent.mp4
Alarm segments: [(6.673153153153153, 8.231351351351352), (9.34918918918919, 12.533333333333333)]
Meters per pixel used: 0.019444444444444445
Moviepy - Building video /content/final_one_video_emv_pcu_WITH_ALARM.mp4.
MoviePy - Writing audio in final_one_video_emv_pcu_WITH_ALARMTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/final_one_video_emv_pcu_WITH_ALARM.mp4



Moviepy - Done !
Moviepy - video ready /content/final_one_video_emv_pcu_WITH_ALARM.mp4
Final video with alarm saved: /content/final_one_video_emv_pcu_WITH_ALARM.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
###

# FINAL ONE-VIDEO CODE:
# Vehicle Detection + Distance + PCU + Ambulance Alert + Alarm Sound in Downloaded Video

!pip install -q ultralytics moviepy scipy
!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/

from ultralytics import YOLO
import cv2
import os
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO_PATH = "/content/WhatsApp2.mp4"
OUTPUT_PATH = "/content/final_one_video_emv_pcu_silent.mp4"
FINAL_OUTPUT_PATH = "/content/final_one_video_emv_pcu_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4
EMV_LOCK_FRAMES = 45

PCU_SMOOTH_WINDOW = 10
NORMAL_HOLD_FRAMES = 30

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. VIDEO
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 10

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

emv_confirm = 0
emv_locked = False
emv_lock_counter = 0
alarm_triggered = False

hold_counter = 0
current_mode = "NORMAL PCU MODE"
frame_idx = 0

# alarm time memory
alarm_segments = []
alarm_active = False
alarm_start_time = None

# -----------------------------
# 4. MAIN LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    # -----------------------------
    # 5. PCU + EMV CONFIRMATION
    # -----------------------------
    pcu_history.append(lane_pcu)
    smooth_pcu = sum(pcu_history) / max(1, len(pcu_history))

    if emv_candidate:
        emv_confirm += 1
    else:
        emv_confirm = 0

    emv_detected = emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 6. DECISION ENGINE + ALARM SEGMENTS
    # -----------------------------
    if emv_detected:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        current_time = frame_idx / fps

        if not alarm_active:
            alarm_start_time = current_time
            alarm_active = True

        if not alarm_triggered:
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        decision = "EMV DETECTED -> CLEAR ROAD"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        decision = "LOCKED EMV PRIORITY"
        emv_lock_counter -= 1

        current_time = frame_idx / fps
        if not alarm_active:
            alarm_start_time = current_time
            alarm_active = True

    else:
        if alarm_active:
            alarm_end_time = frame_idx / fps
            alarm_segments.append((alarm_start_time, alarm_end_time))
            alarm_active = False
            alarm_start_time = None

        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        alarm_triggered = False

        if hold_counter <= 0:
            if smooth_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif smooth_pcu < 5:
                decision = "LOW TRAFFIC -> GREEN 10s"
            elif smooth_pcu < 10:
                decision = "MEDIUM TRAFFIC -> GREEN 20s"
            else:
                decision = "HIGH TRAFFIC -> GREEN 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            decision = "HOLD GREEN"
            hold_counter -= 1

    # -----------------------------
    # 7. DISPLAY
    # -----------------------------
    total_count = sum(lane_counts.values())
    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        cv2.rectangle(frame, (0, 0), (w, 90), (0, 0, 255), -1)
        cv2.putText(frame, "EMERGENCY ALERT - CLEAR ROAD",
                    (20, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (255, 255, 255),
                    3)

    cv2.putText(frame, f"MODE: {current_mode}", (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, decision, (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, f"PCU: {smooth_pcu:.1f}", (20, 190),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {total_count}", (20, 220),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.putText(frame, f"EMV Confirm: {emv_confirm}/{EMV_CONFIRM_FRAMES}", (20, 250),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    if emv_detected and nearest_emv_distance is not None:
        cv2.putText(frame, f"EMV Distance: {nearest_emv_distance:.1f}m", (20, 280),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    y_text = 315
    for cls_name, cnt in sorted(lane_counts.items()):
        cv2.putText(frame, f"{cls_name}: {cnt}", (20, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        y_text += 25

    out.write(frame)

cap.release()
out.release()

# If video ended while alarm still active
if alarm_active:
    alarm_segments.append((alarm_start_time, frame_idx / fps))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)
write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD ALARM AUDIO ONLY DURING EMV SEGMENTS
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 1.0

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

# -----------------------------
# 10. DOWNLOAD
# -----------------------------
from google.colab import files
files.download(FINAL_OUTPUT_PATH)

In [ ]:
# FINAL TWO-VIDEO CODE:
# 2 Lane Detection + Distance + PCU + Ambulance Alert + Stable Alarm

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp3.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/WhatsApp2.mp4"   # Lane 1
VIDEO2_PATH = "/content/WhatsApp3.mp4"   # Lane 2

OUTPUT_PATH = "/content/two_lane_emv_pcu_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_emv_pcu_WITH_STABLE_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10
NORMAL_HOLD_FRAMES = 30

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

OUTPUT_FPS = 10

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")

if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 100

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"

hold_counter = 0
emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

frame_idx = 0

# stable alarm memory
alarm_started = False
alarm_start_time = None
alarm_segments = []

# label smoothing memory
lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. LANE PROCESSOR
# -----------------------------
def process_lane(frame, label_memory, lane_title):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            # ---------------------------------------
            # AMBULANCE LABEL LOCKING
            # ---------------------------------------
            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {
                    "label": label,
                    "lock": 0
                }

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------
# 5. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(
        frame1, lane1_label_memory, "LANE 1"
    )

    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(
        frame2, lane2_label_memory, "LANE 2"
    )

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    if emv_candidate1:
        lane1_emv_confirm += 1
    else:
        lane1_emv_confirm = max(0, lane1_emv_confirm - 1)

    if emv_candidate2:
        lane2_emv_confirm += 1
    else:
        lane2_emv_confirm = max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 6. DECISION ENGINE
    # -----------------------------
    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        if emv1 and not emv2:
            emv_locked_lane = "LANE 1"
        elif emv2 and not emv1:
            emv_locked_lane = "LANE 2"
        else:
            d1 = emv_dist1 if emv_dist1 is not None else 1e9
            d2 = emv_dist2 if emv_dist2 is not None else 1e9
            emv_locked_lane = "LANE 1" if d1 <= d2 else "LANE 2"

        current_green_lane = emv_locked_lane
        decision = f"EMV DETECTED -> CLEAR {current_green_lane}"

        current_time = frame_idx / OUTPUT_FPS

        if not alarm_started:
            alarm_started = True
            alarm_start_time = current_time
            print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        current_green_lane = emv_locked_lane
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane}"
        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        emv_locked_lane = None

        if hold_counter <= 0:
            if smooth_pcu1 > smooth_pcu2:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1
            elif smooth_pcu2 > smooth_pcu1:
                current_green_lane = "LANE 2"
                max_pcu = smooth_pcu2
            else:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1

            if max_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif max_pcu < 5:
                decision = f"GREEN -> {current_green_lane} for 10s"
            elif max_pcu < 10:
                decision = f"GREEN -> {current_green_lane} for 20s"
            else:
                decision = f"GREEN -> {current_green_lane} for 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            decision = f"HOLD GREEN -> {current_green_lane}"
            hold_counter -= 1

    # -----------------------------
    # 7. DISPLAY PANELS
    # -----------------------------
    def draw_lane_panel(frame, lane_name, is_green, pcu_value, counts, emv_flag, emv_confirm):
        count_value = sum(counts.values())

        red_color = (60, 60, 60) if is_green else (0, 0, 255)
        green_color = (0, 255, 0) if is_green else (60, 60, 60)

        cv2.circle(frame, (30, 25), 12, red_color, -1)
        cv2.circle(frame, (65, 25), 12, green_color, -1)

        cv2.putText(frame, lane_name, (90, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        cv2.putText(frame, f"Count: {count_value}", (20, 95),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        cv2.putText(frame, f"EMV: {'YES' if emv_flag else 'NO'} ({emv_confirm})", (20, 125),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    draw_lane_panel(
        frame1,
        "LANE 1",
        current_green_lane == "LANE 1",
        smooth_pcu1,
        counts1,
        emv1,
        lane1_emv_confirm
    )

    draw_lane_panel(
        frame2,
        "LANE 2",
        current_green_lane == "LANE 2",
        smooth_pcu2,
        counts2,
        emv2,
        lane2_emv_confirm
    )

    if current_green_lane == "LANE 1":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 255), 5)
    else:
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 255), 5)

    top = np.zeros((100, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        top[:] = (0, 0, 180)

    cv2.putText(top, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    cv2.putText(top, decision, (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

# stable alarm: from first ambulance detection until video ends
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

# -----------------------------
# 10. DOWNLOAD
# -----------------------------
from google.colab import files
files.download(FINAL_OUTPUT_PATH)

In [ ]:
# FINAL TWO-VIDEO CODE:
# 2 Lane Detection + Distance + PCU + Ambulance Alert + Stable Alarm

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/WhatsApp15.mp4"   # Lane 1
VIDEO2_PATH = "/content/WhatsApp2.mp4"   # Lane 2

OUTPUT_PATH = "/content/two_lane_emv_pcu_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_emv_pcu_WITH_STABLE_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10
NORMAL_HOLD_FRAMES = 30

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

OUTPUT_FPS = 10

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")

if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 100

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"

hold_counter = 0
emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

frame_idx = 0

# stable alarm memory
alarm_started = False
alarm_start_time = None
alarm_segments = []

# label smoothing memory
lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. LANE PROCESSOR
# -----------------------------
def process_lane(frame, label_memory, lane_title):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            # ---------------------------------------
            # AMBULANCE LABEL LOCKING
            # ---------------------------------------
            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {
                    "label": label,
                    "lock": 0
                }

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------
# 5. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(
        frame1, lane1_label_memory, "LANE 1"
    )

    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(
        frame2, lane2_label_memory, "LANE 2"
    )

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    if emv_candidate1:
        lane1_emv_confirm += 1
    else:
        lane1_emv_confirm = max(0, lane1_emv_confirm - 1)

    if emv_candidate2:
        lane2_emv_confirm += 1
    else:
        lane2_emv_confirm = max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 6. DECISION ENGINE
    # -----------------------------
    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        if emv1 and not emv2:
            emv_locked_lane = "LANE 1"
        elif emv2 and not emv1:
            emv_locked_lane = "LANE 2"
        else:
            d1 = emv_dist1 if emv_dist1 is not None else 1e9
            d2 = emv_dist2 if emv_dist2 is not None else 1e9
            emv_locked_lane = "LANE 1" if d1 <= d2 else "LANE 2"

        current_green_lane = emv_locked_lane
        decision = f"EMV DETECTED -> CLEAR {current_green_lane}"

        current_time = frame_idx / OUTPUT_FPS

        if not alarm_started:
            alarm_started = True
            alarm_start_time = current_time
            print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        current_green_lane = emv_locked_lane
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane}"
        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        emv_locked_lane = None

        if hold_counter <= 0:
            if smooth_pcu1 > smooth_pcu2:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1
            elif smooth_pcu2 > smooth_pcu1:
                current_green_lane = "LANE 2"
                max_pcu = smooth_pcu2
            else:
                current_green_lane = "LANE 1"
                max_pcu = smooth_pcu1

            if max_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif max_pcu < 5:
                decision = f"GREEN -> {current_green_lane} for 10s"
            elif max_pcu < 10:
                decision = f"GREEN -> {current_green_lane} for 20s"
            else:
                decision = f"GREEN -> {current_green_lane} for 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            decision = f"HOLD GREEN -> {current_green_lane}"
            hold_counter -= 1

    # -----------------------------
    # 7. DISPLAY PANELS
    # -----------------------------
    def draw_lane_panel(frame, lane_name, is_green, pcu_value, counts, emv_flag, emv_confirm):
        count_value = sum(counts.values())

        red_color = (60, 60, 60) if is_green else (0, 0, 255)
        green_color = (0, 255, 0) if is_green else (60, 60, 60)

        cv2.circle(frame, (30, 25), 12, red_color, -1)
        cv2.circle(frame, (65, 25), 12, green_color, -1)

        cv2.putText(frame, lane_name, (90, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        cv2.putText(frame, f"Count: {count_value}", (20, 95),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        cv2.putText(frame, f"EMV: {'YES' if emv_flag else 'NO'} ({emv_confirm})", (20, 125),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    draw_lane_panel(
        frame1,
        "LANE 1",
        current_green_lane == "LANE 1",
        smooth_pcu1,
        counts1,
        emv1,
        lane1_emv_confirm
    )

    draw_lane_panel(
        frame2,
        "LANE 2",
        current_green_lane == "LANE 2",
        smooth_pcu2,
        counts2,
        emv2,
        lane2_emv_confirm
    )

    if current_green_lane == "LANE 1":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 255), 5)
    else:
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 255), 5)

    top = np.zeros((100, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        top[:] = (0, 0, 180)

    cv2.putText(top, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    cv2.putText(top, decision, (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

# stable alarm: from first ambulance detection until video ends
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

# -----------------------------
# 10. DOWNLOAD
# -----------------------------
from google.colab import files
files.download(FINAL_OUTPUT_PATH)

In [ ]:
# FINAL TWO-VIDEO CODE:
# 2 Lane Detection + Distance + PCU + Ambulance Alert + Stable Alarm
# + Countdown Timer + Green-Yellow-Red Transition

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp16.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/WhatsApp15.mp4"
VIDEO2_PATH = "/content/WhatsApp16.mp4"

OUTPUT_PATH = "/content/two_lane_signal_timer_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_signal_timer_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10

OUTPUT_FPS = 10
YELLOW_SECONDS = 3

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")

if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 120

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
previous_green_lane = "LANE 1"

phase = "GREEN"
phase_counter = 0
planned_green_lane = "LANE 1"

emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

frame_idx = 0

alarm_started = False
alarm_start_time = None
alarm_segments = []

lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. LANE PROCESSOR
# -----------------------------
def process_lane(frame, label_memory):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {"label": label, "lock": 0}

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------
# 5. HELPER FUNCTIONS
# -----------------------------
def green_time_from_pcu(max_pcu):
    if max_pcu == 0:
        return 0
    elif max_pcu < 5:
        return 10
    elif max_pcu < 10:
        return 20
    else:
        return 30

def draw_lane_panel(frame, lane_name, signal_state, pcu_value, counts, emv_flag, emv_confirm, timer_seconds):
    count_value = sum(counts.values())

    red_color = (0, 0, 255) if signal_state == "RED" else (60, 60, 60)
    yellow_color = (0, 255, 255) if signal_state == "YELLOW" else (60, 60, 60)
    green_color = (0, 255, 0) if signal_state == "GREEN" else (60, 60, 60)

    cv2.circle(frame, (25, 25), 11, red_color, -1)
    cv2.circle(frame, (55, 25), 11, yellow_color, -1)
    cv2.circle(frame, (85, 25), 11, green_color, -1)

    cv2.putText(frame, lane_name, (115, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Signal: {signal_state}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Timer: {timer_seconds}s", (20, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Count: {count_value}", (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    cv2.putText(frame, f"EMV: {'YES' if emv_flag else 'NO'} ({emv_confirm})", (20, 185),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

# -----------------------------
# 6. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1, lane1_label_memory)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2, lane2_label_memory)

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else max(0, lane1_emv_confirm - 1)
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 7. DECISION ENGINE WITH TIMER
    # -----------------------------
    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        phase = "GREEN"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        if emv1 and not emv2:
            emv_locked_lane = "LANE 1"
        elif emv2 and not emv1:
            emv_locked_lane = "LANE 2"
        else:
            d1 = emv_dist1 if emv_dist1 is not None else 1e9
            d2 = emv_dist2 if emv_dist2 is not None else 1e9
            emv_locked_lane = "LANE 1" if d1 <= d2 else "LANE 2"

        current_green_lane = emv_locked_lane
        previous_green_lane = current_green_lane

        remaining_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"EMV DETECTED -> CLEAR {current_green_lane} for {remaining_seconds}s"

        current_time = frame_idx / OUTPUT_FPS
        if not alarm_started:
            alarm_started = True
            alarm_start_time = current_time
            print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        current_green_lane = emv_locked_lane
        phase = "GREEN"

        remaining_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane} for {remaining_seconds}s"

        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        emv_locked_lane = None

        if phase_counter <= 0:
            if phase == "GREEN":
                if smooth_pcu1 > smooth_pcu2:
                    next_green_lane = "LANE 1"
                    max_pcu = smooth_pcu1
                elif smooth_pcu2 > smooth_pcu1:
                    next_green_lane = "LANE 2"
                    max_pcu = smooth_pcu2
                else:
                    next_green_lane = "LANE 1"
                    max_pcu = smooth_pcu1

                green_seconds = green_time_from_pcu(max_pcu)

                if green_seconds == 0:
                    current_green_lane = next_green_lane
                    phase = "RED"
                    phase_counter = OUTPUT_FPS
                    decision = "NO TRAFFIC / ALL RED 1s"
                elif next_green_lane != current_green_lane:
                    planned_green_lane = next_green_lane
                    previous_green_lane = current_green_lane
                    phase = "YELLOW"
                    phase_counter = YELLOW_SECONDS * OUTPUT_FPS
                    decision = f"YELLOW -> Switching to {planned_green_lane} in {YELLOW_SECONDS}s"
                else:
                    current_green_lane = next_green_lane
                    phase = "GREEN"
                    phase_counter = green_seconds * OUTPUT_FPS
                    decision = f"GREEN -> {current_green_lane} for {green_seconds}s"

            elif phase == "YELLOW":
                current_green_lane = planned_green_lane

                if current_green_lane == "LANE 1":
                    max_pcu = smooth_pcu1
                else:
                    max_pcu = smooth_pcu2

                green_seconds = green_time_from_pcu(max_pcu)

                phase = "GREEN"
                phase_counter = green_seconds * OUTPUT_FPS
                decision = f"GREEN -> {current_green_lane} for {green_seconds}s"

            elif phase == "RED":
                phase = "GREEN"
                phase_counter = 10 * OUTPUT_FPS
                decision = f"GREEN -> {current_green_lane} for 10s"

        else:
            remaining_seconds = max(1, int(phase_counter / OUTPUT_FPS))

            if phase == "GREEN":
                decision = f"HOLD GREEN -> {current_green_lane} for {remaining_seconds}s"
            elif phase == "YELLOW":
                decision = f"YELLOW -> Switching to {planned_green_lane} in {remaining_seconds}s"
            else:
                decision = f"RED CLEARANCE -> {remaining_seconds}s"

            phase_counter -= 1

    # -----------------------------
    # 8. SIGNAL STATE FOR EACH LANE
    # -----------------------------
    if current_mode == "EMERGENCY OVERRIDE":
        lane1_signal = "GREEN" if current_green_lane == "LANE 1" else "RED"
        lane2_signal = "GREEN" if current_green_lane == "LANE 2" else "RED"
        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))

    else:
        timer_seconds = max(1, int(phase_counter / OUTPUT_FPS))

        if phase == "GREEN":
            lane1_signal = "GREEN" if current_green_lane == "LANE 1" else "RED"
            lane2_signal = "GREEN" if current_green_lane == "LANE 2" else "RED"

        elif phase == "YELLOW":
            lane1_signal = "YELLOW" if previous_green_lane == "LANE 1" else "RED"
            lane2_signal = "YELLOW" if previous_green_lane == "LANE 2" else "RED"

        else:
            lane1_signal = "RED"
            lane2_signal = "RED"

    draw_lane_panel(frame1, "LANE 1", lane1_signal, smooth_pcu1, counts1, emv1, lane1_emv_confirm, timer_seconds)
    draw_lane_panel(frame2, "LANE 2", lane2_signal, smooth_pcu2, counts2, emv2, lane2_emv_confirm, timer_seconds)

    if lane1_signal == "GREEN":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 0), 5)
    elif lane1_signal == "YELLOW":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 255), 5)

    if lane2_signal == "GREEN":
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 0), 5)
    elif lane2_signal == "YELLOW":
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 255), 5)

    top = np.zeros((120, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        top[:] = (0, 0, 180)

    cv2.putText(top, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    cv2.putText(top, decision, (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    cv2.putText(top, f"PHASE: {phase} | TIMER: {timer_seconds}s", (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

# Stable alarm
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 9. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 10. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

from google.colab import files
files.download(FINAL_OUTPUT_PATH)

In [ ]:
# FINAL TWO-VIDEO CODE:
# 2 Lane Detection + Distance + PCU + Ambulance Alert + Stable Alarm
# + Countdown Timer + Green-Yellow-Red Transition
# + PCU-based green time + corrected EMV display + both-lane EMV priority

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/

from ultralytics import YOLO
import math
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/WhatsApp2.mp4"   # Lane 1
VIDEO2_PATH = "/content/WhatsApp15.mp4"   # Lane 2

OUTPUT_PATH = "/content/two_lane_signal_timer_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_signal_timer_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10

OUTPUT_FPS = 10
YELLOW_SECONDS = 3

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")

if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 120

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
previous_green_lane = "LANE 1"

phase = "GREEN"
phase_counter = 0
planned_green_lane = "LANE 1"

emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

frame_idx = 0

alarm_started = False
alarm_start_time = None
alarm_segments = []

lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. LANE PROCESSOR
# -----------------------------
def process_lane(frame, label_memory):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {"label": label, "lock": 0}

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------
# 5. HELPER FUNCTIONS
# -----------------------------
def green_time_from_pcu(pcu):
    if pcu <= 0:
        return 0
    elif pcu < 5:
        return 10
    elif pcu < 10:
        return 20
    elif pcu < 15:
        return 30
    else:
        return 40


def draw_lane_panel(frame, lane_name, signal_state, pcu_value, counts, emv_confirm, timer_seconds):
    count_value = sum(counts.values())

    red_color = (0, 0, 255) if signal_state == "RED" else (60, 60, 60)
    yellow_color = (0, 255, 255) if signal_state == "YELLOW" else (60, 60, 60)
    green_color = (0, 255, 0) if signal_state == "GREEN" else (60, 60, 60)

    cv2.circle(frame, (25, 25), 11, red_color, -1)
    cv2.circle(frame, (55, 25), 11, yellow_color, -1)
    cv2.circle(frame, (85, 25), 11, green_color, -1)

    cv2.putText(frame, lane_name, (115, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Signal: {signal_state}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Timer: {timer_seconds}s", (20, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Count: {count_value}", (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    emv_text = "YES" if emv_confirm > 0 else "NO"
    cv2.putText(frame, f"EMV: {emv_text} ({emv_confirm})", (20, 185),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)


# -----------------------------
# 6. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1, lane1_label_memory)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2, lane2_label_memory)

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else max(0, lane1_emv_confirm - 1)
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # -----------------------------
# 7. DECISION ENGINE WITH PCU-BASED GREEN
# -----------------------------
if emv1 or emv2:
    current_mode = "EMERGENCY OVERRIDE"
    phase = "GREEN"
    emv_locked = True
    emv_lock_counter = EMV_LOCK_FRAMES

    if emv1 and emv2:
        d1 = emv_dist1 if emv_dist1 is not None else 999999
        d2 = emv_dist2 if emv_dist2 is not None else 999999

        if d1 <= d2:
            current_green_lane = "LANE 1"
            reason = "nearest EMV"
        else:
            current_green_lane = "LANE 2"
            reason = "nearest EMV"

    elif emv1:
        current_green_lane = "LANE 1"
        reason = "EMV in Lane 1"

    else:
        current_green_lane = "LANE 2"
        reason = "EMV in Lane 2"

    remaining_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
    decision = f"EMV PRIORITY -> {current_green_lane} for {remaining_seconds}s ({reason})"

    current_time = frame_idx / OUTPUT_FPS
    if not alarm_started:
        alarm_started = True
        alarm_start_time = current_time
        print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

elif emv_locked and emv_lock_counter > 0:
    current_mode = "EMERGENCY OVERRIDE"
    phase = "GREEN"

    remaining_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
    decision = f"LOCKED EMV PRIORITY -> {current_green_lane} for {remaining_seconds}s"
    emv_lock_counter -= 1

else:
    current_mode = "NORMAL PCU MODE"
    emv_locked = False
    emv_locked_lane = None
    phase = "GREEN"

    # PURE PCU-BASED GREEN DECISION
    if smooth_pcu1 > smooth_pcu2:
        current_green_lane = "LANE 1"
        max_pcu = smooth_pcu1
    elif smooth_pcu2 > smooth_pcu1:
        current_green_lane = "LANE 2"
        max_pcu = smooth_pcu2
    else:
        current_green_lane = "LANE 1"
        max_pcu = smooth_pcu1

    green_seconds = green_time_from_pcu(max_pcu)

    if green_seconds == 0:
        decision = "NO TRAFFIC / ALL RED"
        phase = "RED"
    else:
        decision = f"PCU BASED GREEN -> {current_green_lane} for {green_seconds}s | PCU: {max_pcu:.1f}"

    timer_seconds = green_seconds
    # -----------------------------
   # -----------------------------
# 8. SIGNAL STATE FOR EACH LANE
# -----------------------------
if current_mode == "EMERGENCY OVERRIDE":
    lane1_signal = "GREEN" if current_green_lane == "LANE 1" else "RED"
    lane2_signal = "GREEN" if current_green_lane == "LANE 2" else "RED"
    timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))

else:
    if phase == "RED":
        lane1_signal = "RED"
        lane2_signal = "RED"
        timer_seconds = 0
    else:
        lane1_signal = "GREEN" if current_green_lane == "LANE 1" else "RED"
        lane2_signal = "GREEN" if current_green_lane == "LANE 2" else "RED"

# Stable alarm
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 9. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 10. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

from google.colab import files
files.download(FINAL_OUTPUT_PATH)

In [ ]:
# FINAL TWO-VIDEO CODE:
# 2 Lane Detection + Distance + PCU + Ambulance Alert + Stable Alarm
# + Countdown Timer + Green-Yellow-Red Transition
# + PCU-based green time + corrected EMV display + both-lane EMV priority

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/

from ultralytics import YOLO
import math
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/WhatsApp2.mp4"   # Lane 1
VIDEO2_PATH = "/content/WhatsApp15.mp4"   # Lane 2

OUTPUT_PATH = "/content/two_lane_signal_timer_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_signal_timer_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10

OUTPUT_FPS = 10
YELLOW_SECONDS = 3

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")

if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 120

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
previous_green_lane = "LANE 1"

phase = "GREEN"
phase_counter = 0
planned_green_lane = "LANE 1"

emv_locked = False
emv_locked_lane = None
emv_lock_counter = 0

frame_idx = 0

alarm_started = False
alarm_start_time = None
alarm_segments = []

lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. LANE PROCESSOR
# -----------------------------
def process_lane(frame, label_memory):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {"label": label, "lock": 0}

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance

# -----------------------------
# 5. HELPER FUNCTIONS
# -----------------------------
def green_time_from_pcu(pcu):
    if pcu <= 0:
        return 0
    elif pcu < 5:
        return 10
    elif pcu < 10:
        return 20
    elif pcu < 15:
        return 30
    else:
        return 40


def draw_lane_panel(frame, lane_name, signal_state, pcu_value, counts, emv_confirm, timer_seconds):
    count_value = sum(counts.values())

    red_color = (0, 0, 255) if signal_state == "RED" else (60, 60, 60)
    yellow_color = (0, 255, 255) if signal_state == "YELLOW" else (60, 60, 60)
    green_color = (0, 255, 0) if signal_state == "GREEN" else (60, 60, 60)

    cv2.circle(frame, (25, 25), 11, red_color, -1)
    cv2.circle(frame, (55, 25), 11, yellow_color, -1)
    cv2.circle(frame, (85, 25), 11, green_color, -1)

    cv2.putText(frame, lane_name, (115, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Signal: {signal_state}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Timer: {timer_seconds}s", (20, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Count: {count_value}", (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    emv_text = "YES" if emv_confirm > 0 else "NO"
    cv2.putText(frame, f"EMV: {emv_text} ({emv_confirm})", (20, 185),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)


# -----------------------------
# 6. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1, lane1_label_memory)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2, lane2_label_memory)

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else max(0, lane1_emv_confirm - 1)
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # -----------------------------
# 7. DECISION ENGINE WITH PCU-BASED GREEN
# -----------------------------
if emv1 or emv2:
    current_mode = "EMERGENCY OVERRIDE"
    phase = "GREEN"
    emv_locked = True
    emv_lock_counter = EMV_LOCK_FRAMES

    if emv1 and emv2:
        d1 = emv_dist1 if emv_dist1 is not None else 999999
        d2 = emv_dist2 if emv_dist2 is not None else 999999

        if d1 <= d2:
            current_green_lane = "LANE 1"
            reason = "nearest EMV"
        else:
            current_green_lane = "LANE 2"
            reason = "nearest EMV"

    elif emv1:
        current_green_lane = "LANE 1"
        reason = "EMV in Lane 1"

    else:
        current_green_lane = "LANE 2"
        reason = "EMV in Lane 2"

    remaining_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
    decision = f"EMV PRIORITY -> {current_green_lane} for {remaining_seconds}s ({reason})"

    current_time = frame_idx / OUTPUT_FPS
    if not alarm_started:
        alarm_started = True
        alarm_start_time = current_time
        print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

elif emv_locked and emv_lock_counter > 0:
    current_mode = "EMERGENCY OVERRIDE"
    phase = "GREEN"

    remaining_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
    decision = f"LOCKED EMV PRIORITY -> {current_green_lane} for {remaining_seconds}s"
    emv_lock_counter -= 1

else:
    current_mode = "NORMAL PCU MODE"
    emv_locked = False
    emv_locked_lane = None
    phase = "GREEN"

    # PURE PCU-BASED GREEN DECISION
    if smooth_pcu1 > smooth_pcu2:
        current_green_lane = "LANE 1"
        max_pcu = smooth_pcu1
    elif smooth_pcu2 > smooth_pcu1:
        current_green_lane = "LANE 2"
        max_pcu = smooth_pcu2
    else:
        current_green_lane = "LANE 1"
        max_pcu = smooth_pcu1

    green_seconds = green_time_from_pcu(max_pcu)

    if green_seconds == 0:
        decision = "NO TRAFFIC / ALL RED"
        phase = "RED"
    else:
        decision = f"PCU BASED GREEN -> {current_green_lane} for {green_seconds}s | PCU: {max_pcu:.1f}"

    timer_seconds = green_seconds
    # -----------------------------
   # -----------------------------
# 8. SIGNAL STATE FOR EACH LANE
# -----------------------------
if current_mode == "EMERGENCY OVERRIDE":
    lane1_signal = "GREEN" if current_green_lane == "LANE 1" else "RED"
    lane2_signal = "GREEN" if current_green_lane == "LANE 2" else "RED"
    timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))

else:
    if phase == "RED":
        lane1_signal = "RED"
        lane2_signal = "RED"
        timer_seconds = 0
    else:
        lane1_signal = "GREEN" if current_green_lane == "LANE 1" else "RED"
        lane2_signal = "GREEN" if current_green_lane == "LANE 2" else "RED"

# Stable alarm
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 9. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 10. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

from google.colab import files
files.download(FINAL_OUTPUT_PATH)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.4 MB/s eta 0:00:00
cp: cannot stat '/content/drive/MyDrive/New_folder/WhatsApp2.mp4': No such file or directory
cp: cannot stat '/content/drive/MyDrive/New_folder/WhatsApp15.mp4': No such file or directory
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/New_folder/final_best_v8m.pt'

In [ ]:
###
# FINAL TWO-VIDEO CODE:
# PCU-Based Green + Ambulance Detection + Ambulance Count + Stable Alarm

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/WhatsApp2.mp4"
VIDEO2_PATH = "/content/WhatsApp15.mp4"

OUTPUT_PATH = "/content/two_lane_pcu_emv_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_pcu_emv_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 120

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

lane1_total_counts = defaultdict(int)
lane2_total_counts = defaultdict(int)

lane1_counted_ambulance = False
lane2_counted_ambulance = False

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
phase = "GREEN"

emv_locked = False
emv_lock_counter = 0

frame_idx = 0

alarm_started = False
alarm_start_time = None
alarm_segments = []

lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. FUNCTIONS
# -----------------------------
def green_time_from_pcu(pcu):
    if pcu <= 0:
        return 0
    elif pcu < 5:
        return 10
    elif pcu < 10:
        return 20
    elif pcu < 15:
        return 30
    else:
        return 40


def process_lane(frame, label_memory):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {"label": label, "lock": 0}

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance


def draw_lane_panel(frame, lane_name, signal_state, pcu_value, counts, emv_confirm, timer_seconds, total_counts):
    count_value = sum(counts.values())
    ambulance_count = total_counts.get("Ambulance", 0)

    red_color = (0, 0, 255) if signal_state == "RED" else (60, 60, 60)
    green_color = (0, 255, 0) if signal_state == "GREEN" else (60, 60, 60)

    cv2.circle(frame, (25, 25), 12, red_color, -1)
    cv2.circle(frame, (65, 25), 12, green_color, -1)

    cv2.putText(frame, lane_name, (95, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Signal: {signal_state}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Timer: {timer_seconds}s", (20, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {count_value}", (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    emv_text = "YES" if emv_confirm > 0 else "NO"

    cv2.putText(frame, f"EMV: {emv_text} | Ambulance Count: {ambulance_count}",
                (20, 185),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

# -----------------------------
# 5. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1, lane1_label_memory)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2, lane2_label_memory)

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else max(0, lane1_emv_confirm - 1)
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # Count ambulance once per lane after confirmation
    if emv1 and not lane1_counted_ambulance:
        lane1_total_counts["Ambulance"] += 1
        lane1_counted_ambulance = True

    if emv2 and not lane2_counted_ambulance:
        lane2_total_counts["Ambulance"] += 1
        lane2_counted_ambulance = True

    # -----------------------------
    # 6. DECISION ENGINE
    # -----------------------------
    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES
        phase = "GREEN"

        if emv1 and emv2:
            d1 = emv_dist1 if emv_dist1 is not None else 999999
            d2 = emv_dist2 if emv_dist2 is not None else 999999

            if d1 <= d2:
                current_green_lane = "LANE 1"
                reason = "nearest EMV"
            else:
                current_green_lane = "LANE 2"
                reason = "nearest EMV"

        elif emv1:
            current_green_lane = "LANE 1"
            reason = "EMV in Lane 1"

        else:
            current_green_lane = "LANE 2"
            reason = "EMV in Lane 2"

        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"EMV PRIORITY -> {current_green_lane} for {timer_seconds}s ({reason})"

        if not alarm_started:
            alarm_started = True
            alarm_start_time = frame_idx / OUTPUT_FPS
            print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        phase = "GREEN"

        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane} for {timer_seconds}s"
        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        phase = "GREEN"

        if smooth_pcu1 > smooth_pcu2:
            current_green_lane = "LANE 1"
            max_pcu = smooth_pcu1
        elif smooth_pcu2 > smooth_pcu1:
            current_green_lane = "LANE 2"
            max_pcu = smooth_pcu2
        else:
            current_green_lane = "LANE 1"
            max_pcu = smooth_pcu1

        green_seconds = green_time_from_pcu(max_pcu)

        if green_seconds == 0:
            current_green_lane = "NONE"
            timer_seconds = 0
            decision = "NO TRAFFIC / ALL RED"
        else:
            timer_seconds = green_seconds
            decision = f"PCU BASED GREEN -> {current_green_lane} for {green_seconds}s | PCU: {max_pcu:.1f}"

    # -----------------------------
    # 7. SIGNAL STATE
    # -----------------------------
    if current_green_lane == "LANE 1":
        lane1_signal = "GREEN"
        lane2_signal = "RED"
    elif current_green_lane == "LANE 2":
        lane1_signal = "RED"
        lane2_signal = "GREEN"
    else:
        lane1_signal = "RED"
        lane2_signal = "RED"

    draw_lane_panel(
        frame1, "LANE 1", lane1_signal,
        smooth_pcu1, counts1, lane1_emv_confirm,
        timer_seconds, lane1_total_counts
    )

    draw_lane_panel(
        frame2, "LANE 2", lane2_signal,
        smooth_pcu2, counts2, lane2_emv_confirm,
        timer_seconds, lane2_total_counts
    )

    if lane1_signal == "GREEN":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 0), 5)

    if lane2_signal == "GREEN":
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 0), 5)

    top = np.zeros((120, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        top[:] = (0, 0, 180)

    cv2.putText(top, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    cv2.putText(top, decision, (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(top, f"PHASE: {phase} | TIMER: {timer_seconds}s", (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

# Stable alarm from first EMV detection until video end
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)
print("Lane 1 total counts:", dict(lane1_total_counts))
print("Lane 2 total counts:", dict(lane2_total_counts))

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

from google.colab import files
files.download(FINAL_OUTPUT_PATH)

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED
Silent video saved: /content/two_lane_pcu_emv_silent.mp4
Alarm segments: [(58.6, 61.2)]
Meters per pixel used: 0.019444444444444445
Lane 1 total counts: {'Ambulance': 1}
Lane 2 total counts: {}
Moviepy - Building video /content/two_lane_pcu_emv_WITH_ALARM.mp4.
MoviePy - Writing audio in two_lane_pcu_emv_WITH_ALARMTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/two_lane_pcu_emv_WITH_ALARM.mp4



Moviepy - Done !
Moviepy - video ready /content/two_lane_pcu_emv_WITH_ALARM.mp4
Final video with stable alarm saved: /content/two_lane_pcu_emv_WITH_ALARM.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("/content/two_lane_pcu_emv_WITH_ALARM.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
###
# FINAL TWO-VIDEO CODE:
# PCU-Based Green + Ambulance Detection + Ambulance Count + Stable Alarm

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/WhatsApp3.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp15.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/WhatsApp3.mp4"
VIDEO2_PATH = "/content/WhatsApp15.mp4"

OUTPUT_PATH = "/content/two_lane_pcu_emv_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_pcu_emv_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 120

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

lane1_total_counts = defaultdict(int)
lane2_total_counts = defaultdict(int)

lane1_counted_ambulance = False
lane2_counted_ambulance = False

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
phase = "GREEN"

emv_locked = False
emv_lock_counter = 0

frame_idx = 0

alarm_started = False
alarm_start_time = None
alarm_segments = []

lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. FUNCTIONS
# -----------------------------
def green_time_from_pcu(pcu):
    if pcu <= 0:
        return 0
    elif pcu < 5:
        return 10
    elif pcu < 10:
        return 20
    elif pcu < 15:
        return 30
    else:
        return 40


def process_lane(frame, label_memory):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {"label": label, "lock": 0}

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance


def draw_lane_panel(frame, lane_name, signal_state, pcu_value, counts, emv_confirm, timer_seconds, total_counts):
    count_value = sum(counts.values())
    ambulance_count = total_counts.get("Ambulance", 0)

    red_color = (0, 0, 255) if signal_state == "RED" else (60, 60, 60)
    green_color = (0, 255, 0) if signal_state == "GREEN" else (60, 60, 60)

    cv2.circle(frame, (25, 25), 12, red_color, -1)
    cv2.circle(frame, (65, 25), 12, green_color, -1)

    cv2.putText(frame, lane_name, (95, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Signal: {signal_state}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Timer: {timer_seconds}s", (20, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {count_value}", (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    emv_text = "YES" if emv_confirm > 0 else "NO"

    cv2.putText(frame, f"EMV: {emv_text} | Ambulance Count: {ambulance_count}",
                (20, 185),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

# -----------------------------
# 5. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1, lane1_label_memory)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2, lane2_label_memory)

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else max(0, lane1_emv_confirm - 1)
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # Count ambulance once per lane after confirmation
    if emv1 and not lane1_counted_ambulance:
        lane1_total_counts["Ambulance"] += 1
        lane1_counted_ambulance = True

    if emv2 and not lane2_counted_ambulance:
        lane2_total_counts["Ambulance"] += 1
        lane2_counted_ambulance = True

    # -----------------------------
    # 6. DECISION ENGINE
    # -----------------------------
    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES
        phase = "GREEN"

        if emv1 and emv2:
            d1 = emv_dist1 if emv_dist1 is not None else 999999
            d2 = emv_dist2 if emv_dist2 is not None else 999999

            if d1 <= d2:
                current_green_lane = "LANE 1"
                reason = "nearest EMV"
            else:
                current_green_lane = "LANE 2"
                reason = "nearest EMV"

        elif emv1:
            current_green_lane = "LANE 1"
            reason = "EMV in Lane 1"

        else:
            current_green_lane = "LANE 2"
            reason = "EMV in Lane 2"

        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"EMV PRIORITY -> {current_green_lane} for {timer_seconds}s ({reason})"

        if not alarm_started:
            alarm_started = True
            alarm_start_time = frame_idx / OUTPUT_FPS
            print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        phase = "GREEN"

        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane} for {timer_seconds}s"
        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        phase = "GREEN"

        if smooth_pcu1 > smooth_pcu2:
            current_green_lane = "LANE 1"
            max_pcu = smooth_pcu1
        elif smooth_pcu2 > smooth_pcu1:
            current_green_lane = "LANE 2"
            max_pcu = smooth_pcu2
        else:
            current_green_lane = "LANE 1"
            max_pcu = smooth_pcu1

        green_seconds = green_time_from_pcu(max_pcu)

        if green_seconds == 0:
            current_green_lane = "NONE"
            timer_seconds = 0
            decision = "NO TRAFFIC / ALL RED"
        else:
            timer_seconds = green_seconds
            decision = f"PCU BASED GREEN -> {current_green_lane} for {green_seconds}s | PCU: {max_pcu:.1f}"

    # -----------------------------
    # 7. SIGNAL STATE
    # -----------------------------
    if current_green_lane == "LANE 1":
        lane1_signal = "GREEN"
        lane2_signal = "RED"
    elif current_green_lane == "LANE 2":
        lane1_signal = "RED"
        lane2_signal = "GREEN"
    else:
        lane1_signal = "RED"
        lane2_signal = "RED"

    draw_lane_panel(
        frame1, "LANE 1", lane1_signal,
        smooth_pcu1, counts1, lane1_emv_confirm,
        timer_seconds, lane1_total_counts
    )

    draw_lane_panel(
        frame2, "LANE 2", lane2_signal,
        smooth_pcu2, counts2, lane2_emv_confirm,
        timer_seconds, lane2_total_counts
    )

    if lane1_signal == "GREEN":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 0), 5)

    if lane2_signal == "GREEN":
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 0), 5)

    top = np.zeros((120, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        top[:] = (0, 0, 180)

    cv2.putText(top, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    cv2.putText(top, decision, (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(top, f"PHASE: {phase} | TIMER: {timer_seconds}s", (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

# Stable alarm from first EMV detection until video end
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)
print("Lane 1 total counts:", dict(lane1_total_counts))
print("Lane 2 total counts:", dict(lane2_total_counts))

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

from google.colab import files
files.download(FINAL_OUTPUT_PATH)

EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED
Silent video saved: /content/two_lane_pcu_emv_silent.mp4
Alarm segments: [(48.1, 74.4)]
Meters per pixel used: 0.019444444444444445
Lane 1 total counts: {'Ambulance': 1}
Lane 2 total counts: {}
Moviepy - Building video /content/two_lane_pcu_emv_WITH_ALARM.mp4.
MoviePy - Writing audio in two_lane_pcu_emv_WITH_ALARMTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/two_lane_pcu_emv_WITH_ALARM.mp4



Moviepy - Done !
Moviepy - video ready /content/two_lane_pcu_emv_WITH_ALARM.mp4
Final video with stable alarm saved: /content/two_lane_pcu_emv_WITH_ALARM.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
###
# FINAL TWO-VIDEO CODE:
# PCU-Based Green + Ambulance Detection + Ambulance Count + Stable Alarm

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/WhatsApp2.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/WhatsApp8.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/WhatsApp2.mp4"
VIDEO2_PATH = "/content/WhatsApp8.mp4"

OUTPUT_PATH = "/content/two_lane_pcu_emv_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_pcu_emv_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 120

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

lane1_total_counts = defaultdict(int)
lane2_total_counts = defaultdict(int)

lane1_counted_ambulance = False
lane2_counted_ambulance = False

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
phase = "GREEN"

emv_locked = False
emv_lock_counter = 0

frame_idx = 0

alarm_started = False
alarm_start_time = None
alarm_segments = []

lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. FUNCTIONS
# -----------------------------
def green_time_from_pcu(pcu):
    if pcu <= 0:
        return 0
    elif pcu < 5:
        return 10
    elif pcu < 10:
        return 20
    elif pcu < 15:
        return 30
    else:
        return 40


def process_lane(frame, label_memory):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {"label": label, "lock": 0}

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance


def draw_lane_panel(frame, lane_name, signal_state, pcu_value, counts, emv_confirm, timer_seconds, total_counts):
    count_value = sum(counts.values())
    ambulance_count = total_counts.get("Ambulance", 0)

    red_color = (0, 0, 255) if signal_state == "RED" else (60, 60, 60)
    green_color = (0, 255, 0) if signal_state == "GREEN" else (60, 60, 60)

    cv2.circle(frame, (25, 25), 12, red_color, -1)
    cv2.circle(frame, (65, 25), 12, green_color, -1)

    cv2.putText(frame, lane_name, (95, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Signal: {signal_state}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Timer: {timer_seconds}s", (20, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {count_value}", (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    emv_text = "YES" if emv_confirm > 0 else "NO"

    cv2.putText(frame, f"EMV: {emv_text} | Ambulance Count: {ambulance_count}",
                (20, 185),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

# -----------------------------
# 5. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1, lane1_label_memory)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2, lane2_label_memory)

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else max(0, lane1_emv_confirm - 1)
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # Count ambulance once per lane after confirmation
    if emv1 and not lane1_counted_ambulance:
        lane1_total_counts["Ambulance"] += 1
        lane1_counted_ambulance = True

    if emv2 and not lane2_counted_ambulance:
        lane2_total_counts["Ambulance"] += 1
        lane2_counted_ambulance = True

    # -----------------------------
    # 6. DECISION ENGINE
    # -----------------------------
    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES
        phase = "GREEN"

        if emv1 and emv2:
            d1 = emv_dist1 if emv_dist1 is not None else 999999
            d2 = emv_dist2 if emv_dist2 is not None else 999999

            if d1 <= d2:
                current_green_lane = "LANE 1"
                reason = "nearest EMV"
            else:
                current_green_lane = "LANE 2"
                reason = "nearest EMV"

        elif emv1:
            current_green_lane = "LANE 1"
            reason = "EMV in Lane 1"

        else:
            current_green_lane = "LANE 2"
            reason = "EMV in Lane 2"

        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"EMV PRIORITY -> {current_green_lane} for {timer_seconds}s ({reason})"

        if not alarm_started:
            alarm_started = True
            alarm_start_time = frame_idx / OUTPUT_FPS
            print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        phase = "GREEN"

        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane} for {timer_seconds}s"
        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        phase = "GREEN"

        if smooth_pcu1 > smooth_pcu2:
            current_green_lane = "LANE 1"
            max_pcu = smooth_pcu1
        elif smooth_pcu2 > smooth_pcu1:
            current_green_lane = "LANE 2"
            max_pcu = smooth_pcu2
        else:
            current_green_lane = "LANE 1"
            max_pcu = smooth_pcu1

        green_seconds = green_time_from_pcu(max_pcu)

        if green_seconds == 0:
            current_green_lane = "NONE"
            timer_seconds = 0
            decision = "NO TRAFFIC / ALL RED"
        else:
            timer_seconds = green_seconds
            decision = f"PCU BASED GREEN -> {current_green_lane} for {green_seconds}s | PCU: {max_pcu:.1f}"

    # -----------------------------
    # 7. SIGNAL STATE
    # -----------------------------
    if current_green_lane == "LANE 1":
        lane1_signal = "GREEN"
        lane2_signal = "RED"
    elif current_green_lane == "LANE 2":
        lane1_signal = "RED"
        lane2_signal = "GREEN"
    else:
        lane1_signal = "RED"
        lane2_signal = "RED"

    draw_lane_panel(
        frame1, "LANE 1", lane1_signal,
        smooth_pcu1, counts1, lane1_emv_confirm,
        timer_seconds, lane1_total_counts
    )

    draw_lane_panel(
        frame2, "LANE 2", lane2_signal,
        smooth_pcu2, counts2, lane2_emv_confirm,
        timer_seconds, lane2_total_counts
    )

    if lane1_signal == "GREEN":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 0), 5)

    if lane2_signal == "GREEN":
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 0), 5)

    top = np.zeros((120, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        top[:] = (0, 0, 180)

    cv2.putText(top, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    cv2.putText(top, decision, (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(top, f"PHASE: {phase} | TIMER: {timer_seconds}s", (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

# Stable alarm from first EMV detection until video end
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)
print("Lane 1 total counts:", dict(lane1_total_counts))
print("Lane 2 total counts:", dict(lane2_total_counts))

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

from google.colab import files
files.download(FINAL_OUTPUT_PATH)

EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED
Silent video saved: /content/two_lane_pcu_emv_silent.mp4
Alarm segments: [(58.6, 61.2)]
Meters per pixel used: 0.019444444444444445
Lane 1 total counts: {'Ambulance': 1}
Lane 2 total counts: {}
Moviepy - Building video /content/two_lane_pcu_emv_WITH_ALARM.mp4.
MoviePy - Writing audio in two_lane_pcu_emv_WITH_ALARMTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/two_lane_pcu_emv_WITH_ALARM.mp4



Moviepy - Done !
Moviepy - video ready /content/two_lane_pcu_emv_WITH_ALARM.mp4
Final video with stable alarm saved: /content/two_lane_pcu_emv_WITH_ALARM.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
###
# FINAL TWO-VIDEO CODE:
# PCU-Based Green + Ambulance Detection + Ambulance Count + Stable Alarm

!pip install -q ultralytics moviepy scipy

!cp "/content/drive/MyDrive/New_folder/Recording 2.mp4" /content/
!cp "/content/drive/MyDrive/New_folder/Recording 3.mp4" /content/

from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"

VIDEO1_PATH = "/content/Recording 2.mp4"
VIDEO2_PATH = "/content/Recording 3.mp4"

OUTPUT_PATH = "/content/two_lane_pcu_emv_silent.mp4"
FINAL_OUTPUT_PATH = "/content/two_lane_pcu_emv_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.45
EMV_MIN_AREA = 2000
EMV_CONFIRM_FRAMES = 3
EMV_LOCK_FRAMES = 90

PCU_SMOOTH_WINDOW = 10
OUTPUT_FPS = 10

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. OPEN VIDEOS
# -----------------------------
cap1 = cv2.VideoCapture(VIDEO1_PATH)
cap2 = cv2.VideoCapture(VIDEO2_PATH)

if not cap1.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO1_PATH}")
if not cap2.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO2_PATH}")

w1 = int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
h1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))

target_h = min(h1, h2)
target_w1 = int(w1 * target_h / h1)
target_w2 = int(w2 * target_h / h2)

combined_w = target_w1 + target_w2
combined_h = target_h + 120

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    OUTPUT_FPS,
    (combined_w, combined_h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
lane1_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)
lane2_pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

lane1_emv_confirm = 0
lane2_emv_confirm = 0

lane1_total_counts = defaultdict(int)
lane2_total_counts = defaultdict(int)

lane1_counted_ambulance = False
lane2_counted_ambulance = False

current_mode = "NORMAL PCU MODE"
current_green_lane = "LANE 1"
phase = "GREEN"

emv_locked = False
emv_lock_counter = 0

frame_idx = 0

alarm_started = False
alarm_start_time = None
alarm_segments = []

lane1_label_memory = {}
lane2_label_memory = {}
AMBULANCE_LOCK_FRAMES = 90

# -----------------------------
# 4. FUNCTIONS
# -----------------------------
def green_time_from_pcu(pcu):
    if pcu <= 0:
        return 0
    elif pcu < 5:
        return 10
    elif pcu < 10:
        return 20
    elif pcu < 15:
        return 30
    else:
        return 40


def process_lane(frame, label_memory):
    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    h, w = frame.shape[:2]

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue
            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            object_key = f"{int(cx / 60)}_{int(cy / 60)}"

            if object_key not in label_memory:
                label_memory[object_key] = {"label": label, "lock": 0}

            if label == "Ambulance" and conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA:
                label_memory[object_key]["label"] = "Ambulance"
                label_memory[object_key]["lock"] = AMBULANCE_LOCK_FRAMES

            elif label_memory[object_key]["lock"] > 0:
                label = label_memory[object_key]["label"]
                label_memory[object_key]["lock"] -= 1

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label == "Ambulance"
                and (
                    (conf >= EMV_CONF_THRES and area >= EMV_MIN_AREA)
                    or label_memory[object_key]["lock"] > 0
                )
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if label == "Ambulance":
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    return frame, lane_counts, lane_pcu, emv_candidate, nearest_emv_distance


def draw_lane_panel(frame, lane_name, signal_state, pcu_value, counts, emv_confirm, timer_seconds, total_counts):
    count_value = sum(counts.values())
    ambulance_count = total_counts.get("Ambulance", 0)

    red_color = (0, 0, 255) if signal_state == "RED" else (60, 60, 60)
    green_color = (0, 255, 0) if signal_state == "GREEN" else (60, 60, 60)

    cv2.circle(frame, (25, 25), 12, red_color, -1)
    cv2.circle(frame, (65, 25), 12, green_color, -1)

    cv2.putText(frame, lane_name, (95, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Signal: {signal_state}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Timer: {timer_seconds}s", (20, 95),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.putText(frame, f"PCU: {pcu_value:.1f}", (20, 125),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {count_value}", (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    emv_text = "YES" if emv_confirm > 0 else "NO"

    cv2.putText(frame, f"EMV: {emv_text} | Ambulance Count: {ambulance_count}",
                (20, 185),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

# -----------------------------
# 5. MAIN LOOP
# -----------------------------
while True:
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        break

    frame_idx += 1

    frame1 = cv2.resize(frame1, (target_w1, target_h))
    frame2 = cv2.resize(frame2, (target_w2, target_h))

    frame1, counts1, pcu1, emv_candidate1, emv_dist1 = process_lane(frame1, lane1_label_memory)
    frame2, counts2, pcu2, emv_candidate2, emv_dist2 = process_lane(frame2, lane2_label_memory)

    lane1_pcu_history.append(pcu1)
    lane2_pcu_history.append(pcu2)

    smooth_pcu1 = sum(lane1_pcu_history) / max(1, len(lane1_pcu_history))
    smooth_pcu2 = sum(lane2_pcu_history) / max(1, len(lane2_pcu_history))

    lane1_emv_confirm = lane1_emv_confirm + 1 if emv_candidate1 else max(0, lane1_emv_confirm - 1)
    lane2_emv_confirm = lane2_emv_confirm + 1 if emv_candidate2 else max(0, lane2_emv_confirm - 1)

    emv1 = lane1_emv_confirm >= EMV_CONFIRM_FRAMES
    emv2 = lane2_emv_confirm >= EMV_CONFIRM_FRAMES

    # Count ambulance once per lane after confirmation
    if emv1 and not lane1_counted_ambulance:
        lane1_total_counts["Ambulance"] += 1
        lane1_counted_ambulance = True

    if emv2 and not lane2_counted_ambulance:
        lane2_total_counts["Ambulance"] += 1
        lane2_counted_ambulance = True

    # -----------------------------
    # 6. DECISION ENGINE
    # -----------------------------
    if emv1 or emv2:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES
        phase = "GREEN"

        if emv1 and emv2:
            d1 = emv_dist1 if emv_dist1 is not None else 999999
            d2 = emv_dist2 if emv_dist2 is not None else 999999

            if d1 <= d2:
                current_green_lane = "LANE 1"
                reason = "nearest EMV"
            else:
                current_green_lane = "LANE 2"
                reason = "nearest EMV"

        elif emv1:
            current_green_lane = "LANE 1"
            reason = "EMV in Lane 1"

        else:
            current_green_lane = "LANE 2"
            reason = "EMV in Lane 2"

        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"EMV PRIORITY -> {current_green_lane} for {timer_seconds}s ({reason})"

        if not alarm_started:
            alarm_started = True
            alarm_start_time = frame_idx / OUTPUT_FPS
            print("EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED")

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        phase = "GREEN"

        timer_seconds = max(1, int(emv_lock_counter / OUTPUT_FPS))
        decision = f"LOCKED EMV PRIORITY -> {current_green_lane} for {timer_seconds}s"
        emv_lock_counter -= 1

    else:
        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        phase = "GREEN"

        if smooth_pcu1 > smooth_pcu2:
            current_green_lane = "LANE 1"
            max_pcu = smooth_pcu1
        elif smooth_pcu2 > smooth_pcu1:
            current_green_lane = "LANE 2"
            max_pcu = smooth_pcu2
        else:
            current_green_lane = "LANE 1"
            max_pcu = smooth_pcu1

        green_seconds = green_time_from_pcu(max_pcu)

        if green_seconds == 0:
            current_green_lane = "NONE"
            timer_seconds = 0
            decision = "NO TRAFFIC / ALL RED"
        else:
            timer_seconds = green_seconds
            decision = f"PCU BASED GREEN -> {current_green_lane} for {green_seconds}s | PCU: {max_pcu:.1f}"

    # -----------------------------
    # 7. SIGNAL STATE
    # -----------------------------
    if current_green_lane == "LANE 1":
        lane1_signal = "GREEN"
        lane2_signal = "RED"
    elif current_green_lane == "LANE 2":
        lane1_signal = "RED"
        lane2_signal = "GREEN"
    else:
        lane1_signal = "RED"
        lane2_signal = "RED"

    draw_lane_panel(
        frame1, "LANE 1", lane1_signal,
        smooth_pcu1, counts1, lane1_emv_confirm,
        timer_seconds, lane1_total_counts
    )

    draw_lane_panel(
        frame2, "LANE 2", lane2_signal,
        smooth_pcu2, counts2, lane2_emv_confirm,
        timer_seconds, lane2_total_counts
    )

    if lane1_signal == "GREEN":
        cv2.rectangle(frame1, (0, 0), (target_w1 - 1, target_h - 1), (0, 255, 0), 5)

    if lane2_signal == "GREEN":
        cv2.rectangle(frame2, (0, 0), (target_w2 - 1, target_h - 1), (0, 255, 0), 5)

    top = np.zeros((120, combined_w, 3), dtype=np.uint8)

    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        top[:] = (0, 0, 180)

    cv2.putText(top, f"MODE: {current_mode}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, mode_color, 2)

    cv2.putText(top, decision, (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(top, f"PHASE: {phase} | TIMER: {timer_seconds}s", (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    bottom = np.hstack([frame1, frame2])
    combined = np.vstack([top, bottom])

    out.write(combined)

cap1.release()
cap2.release()
out.release()

# Stable alarm from first EMV detection until video end
if alarm_started and alarm_start_time is not None:
    alarm_segments.append((alarm_start_time, frame_idx / OUTPUT_FPS))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)
print("Lane 1 total counts:", dict(lane1_total_counts))
print("Lane 2 total counts:", dict(lane2_total_counts))

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)

write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD STABLE ALARM AUDIO
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 0.5

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with stable alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

from google.colab import files
files.download(FINAL_OUTPUT_PATH)

EMERGENCY VEHICLE DETECTED - STABLE ALARM STARTED
Silent video saved: /content/two_lane_pcu_emv_silent.mp4
Alarm segments: [(5.0, 36.4)]
Meters per pixel used: 0.019444444444444445
Lane 1 total counts: {'Ambulance': 1}
Lane 2 total counts: {}
Moviepy - Building video /content/two_lane_pcu_emv_WITH_ALARM.mp4.
MoviePy - Writing audio in two_lane_pcu_emv_WITH_ALARMTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/two_lane_pcu_emv_WITH_ALARM.mp4



Moviepy - Done !
Moviepy - video ready /content/two_lane_pcu_emv_WITH_ALARM.mp4
Final video with stable alarm saved: /content/two_lane_pcu_emv_WITH_ALARM.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
###

# FINAL ONE-VIDEO CODE:
# Vehicle Detection + Distance + PCU + Ambulance Alert + Alarm Sound in Downloaded Video

!pip install -q ultralytics moviepy scipy
!cp "/content/drive/MyDrive/New_folder/Recording 3.mp4" /content/

from ultralytics import YOLO
import cv2
import os
import numpy as np
from collections import defaultdict, deque
from scipy.io.wavfile import write
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# -----------------------------
# 1. SETTINGS
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/New_folder/final_best_v8m.pt"
VIDEO_PATH = "/content/Recording 3.mp4"
OUTPUT_PATH = "/content/final_one_video_emv_pcu_silent.mp4"
FINAL_OUTPUT_PATH = "/content/final_one_video_emv_pcu_WITH_ALARM.mp4"

model = YOLO(MODEL_PATH)

allowed_classes = {
    "Ambulance", "Auto_ricksaw", "Bike", "Bus", "Fire_truck",
    "Leguna", "Pickup", "Police_car", "Private_car", "Truck", "cng"
}

emergency_classes = {"Ambulance"}

pcu_map = {
    "Bike": 0.5,
    "Auto_ricksaw": 0.8,
    "cng": 1.0,
    "Private_car": 1.0,
    "Pickup": 1.2,
    "Leguna": 1.5,
    "Bus": 3.0,
    "Truck": 3.0,
    "Ambulance": 1.0,
    "Police_car": 1.0,
    "Fire_truck": 3.0
}

CONF_THRES = 0.20
MIN_BOX_AREA = 500

EMV_CONF_THRES = 0.55
EMV_MIN_AREA = 2500
EMV_CONFIRM_FRAMES = 4
EMV_LOCK_FRAMES = 45

PCU_SMOOTH_WINDOW = 10
NORMAL_HOLD_FRAMES = 30

known_real_distance_m = 3.5
known_pixel_distance = 180
meters_per_pixel = known_real_distance_m / known_pixel_distance

# -----------------------------
# 2. VIDEO
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 10

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

# -----------------------------
# 3. MEMORY
# -----------------------------
pcu_history = deque(maxlen=PCU_SMOOTH_WINDOW)

emv_confirm = 0
emv_locked = False
emv_lock_counter = 0
alarm_triggered = False

hold_counter = 0
current_mode = "NORMAL PCU MODE"
frame_idx = 0

# alarm time memory
alarm_segments = []
alarm_active = False
alarm_start_time = None

# -----------------------------
# 4. MAIN LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    lane_counts = defaultdict(int)
    lane_pcu = 0.0
    emv_candidate = False
    nearest_emv_distance = None

    results = model.predict(
        frame,
        conf=CONF_THRES,
        imgsz=960,
        verbose=False
    )

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        names = results[0].names

        for box, cls_id, conf in zip(boxes, cls_ids, confs):
            x1, y1, x2, y2 = map(int, box)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            label = names[int(cls_id)]

            if label not in allowed_classes:
                continue

            if area < MIN_BOX_AREA:
                continue

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            pixel_distance = abs(h - cy)
            meter_distance = pixel_distance * meters_per_pixel

            lane_counts[label] += 1
            lane_pcu += pcu_map.get(label, 1.0)

            is_emv_candidate = (
                label in emergency_classes
                and conf >= EMV_CONF_THRES
                and area >= EMV_MIN_AREA
            )

            if is_emv_candidate:
                emv_candidate = True
                if nearest_emv_distance is None or meter_distance < nearest_emv_distance:
                    nearest_emv_distance = meter_distance

            color = (0, 255, 0)
            if is_emv_candidate:
                color = (0, 165, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            cv2.putText(
                frame,
                f"{label} {conf:.2f} {meter_distance:.1f}m",
                (x1, max(25, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    # -----------------------------
    # 5. PCU + EMV CONFIRMATION
    # -----------------------------
    pcu_history.append(lane_pcu)
    smooth_pcu = sum(pcu_history) / max(1, len(pcu_history))

    if emv_candidate:
        emv_confirm += 1
    else:
        emv_confirm = 0

    emv_detected = emv_confirm >= EMV_CONFIRM_FRAMES

    # -----------------------------
    # 6. DECISION ENGINE + ALARM SEGMENTS
    # -----------------------------
    if emv_detected:
        current_mode = "EMERGENCY OVERRIDE"
        emv_locked = True
        emv_lock_counter = EMV_LOCK_FRAMES

        current_time = frame_idx / fps

        if not alarm_active:
            alarm_start_time = current_time
            alarm_active = True

        if not alarm_triggered:
            print("EMERGENCY VEHICLE DETECTED - CLEAR ROAD")
            alarm_triggered = True

        decision = "EMV DETECTED -> CLEAR ROAD"

    elif emv_locked and emv_lock_counter > 0:
        current_mode = "EMERGENCY OVERRIDE"
        decision = "LOCKED EMV PRIORITY"
        emv_lock_counter -= 1

        current_time = frame_idx / fps
        if not alarm_active:
            alarm_start_time = current_time
            alarm_active = True

    else:
        if alarm_active:
            alarm_end_time = frame_idx / fps
            alarm_segments.append((alarm_start_time, alarm_end_time))
            alarm_active = False
            alarm_start_time = None

        current_mode = "NORMAL PCU MODE"
        emv_locked = False
        alarm_triggered = False

        if hold_counter <= 0:
            if smooth_pcu == 0:
                decision = "NO TRAFFIC / SKIP"
            elif smooth_pcu < 5:
                decision = "LOW TRAFFIC -> GREEN 10s"
            elif smooth_pcu < 10:
                decision = "MEDIUM TRAFFIC -> GREEN 20s"
            else:
                decision = "HIGH TRAFFIC -> GREEN 30s"

            hold_counter = NORMAL_HOLD_FRAMES

        else:
            decision = "HOLD GREEN"
            hold_counter -= 1

    # -----------------------------
    # 7. DISPLAY
    # -----------------------------
    total_count = sum(lane_counts.values())
    mode_color = (0, 0, 255) if current_mode == "EMERGENCY OVERRIDE" else (0, 255, 255)

    if current_mode == "EMERGENCY OVERRIDE":
        cv2.rectangle(frame, (0, 0), (w, 90), (0, 0, 255), -1)
        cv2.putText(frame, "EMERGENCY ALERT - CLEAR ROAD",
                    (20, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (255, 255, 255),
                    3)

    cv2.putText(frame, f"MODE: {current_mode}", (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, decision, (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, mode_color, 2)

    cv2.putText(frame, f"PCU: {smooth_pcu:.1f}", (20, 190),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Vehicle Count: {total_count}", (20, 220),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cv2.putText(frame, f"EMV Confirm: {emv_confirm}/{EMV_CONFIRM_FRAMES}", (20, 250),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    if emv_detected and nearest_emv_distance is not None:
        cv2.putText(frame, f"EMV Distance: {nearest_emv_distance:.1f}m", (20, 280),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

    y_text = 315
    for cls_name, cnt in sorted(lane_counts.items()):
        cv2.putText(frame, f"{cls_name}: {cnt}", (20, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        y_text += 25

    out.write(frame)

cap.release()
out.release()

# If video ended while alarm still active
if alarm_active:
    alarm_segments.append((alarm_start_time, frame_idx / fps))

print("Silent video saved:", OUTPUT_PATH)
print("Alarm segments:", alarm_segments)
print("Meters per pixel used:", meters_per_pixel)

# -----------------------------
# 8. CREATE BEEP AUDIO
# -----------------------------
sample_rate = 44100
duration = 0.5
frequency = 1000

t = np.linspace(0, duration, int(sample_rate * duration), False)
beep = 0.5 * np.sin(2 * np.pi * frequency * t)
write("/content/beep.wav", sample_rate, beep.astype(np.float32))

# -----------------------------
# 9. ADD ALARM AUDIO ONLY DURING EMV SEGMENTS
# -----------------------------
video = VideoFileClip(OUTPUT_PATH)
audio_clips = []

for start, end in alarm_segments:
    current = start
    while current < end:
        beep_clip = AudioFileClip("/content/beep.wav").set_start(current)
        audio_clips.append(beep_clip)
        current += 1.0

if audio_clips:
    final_audio = CompositeAudioClip(audio_clips)
    final_video = video.set_audio(final_audio)

    final_video.write_videofile(
        FINAL_OUTPUT_PATH,
        codec="libx264",
        audio_codec="aac"
    )

    print("Final video with alarm saved:", FINAL_OUTPUT_PATH)
else:
    FINAL_OUTPUT_PATH = OUTPUT_PATH
    print("No EMV detected, so no alarm audio added.")

# -----------------------------
# 10. DOWNLOAD
# -----------------------------
from google.colab import files
files.download(FINAL_OUTPUT_PATH)

Silent video saved: /content/final_one_video_emv_pcu_silent.mp4
Alarm segments: []
Meters per pixel used: 0.019444444444444445
No EMV detected, so no alarm audio added.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>